# GMA / FDIIR — Exploração do Espaço de Soluções

**Estudo Dirigido — Mestrado FDIIR**  

Execução dos quatro métodos de seleção morfológica sobre a CCM:  
Scoring linear ponderado · Ideal point · Pareto-based MA · HMMD  



## Setup

In [104]:
import numpy as np
import pandas as pd
import itertools
from scipy.stats import spearmanr

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.3f}'.format)

### Dados

As constantes do projeto.


#### Parâmetros morfológicos e design alternatives (DAs)

In [105]:
# Módulos e seus valores (IDs internos)
MODULES = ['D', 'I', 'ID', 'R']

MODULE_VALUES = {
    'D':  ['D1', 'D2', 'D3', 'D4'],
    'I':  ['I1', 'I2', 'I3', 'I4'],
    'ID': ['ID1', 'ID2', 'ID3'],
    'R':  ['R1', 'R2', 'R3']
}

# Nomes descritivos das DAs
DA_NAMES = {
    'D1':  'model-based',
    'D2':  'signal-based',
    'D3':  'data-driven',
    'D4':  'active diagnosis',
    'I1':  'output comparison',
    'I2':  'voting',
    'I3':  'dependency modeling',
    'I4':  'adaptive filters',
    'ID1': 'rule-based',
    'ID2': 'statistical',
    'ID3': 'ML identification',
    'R1':  'exclusion',
    'R2':  'weight reconfiguration',
    'R3':  'bypass'
}

# Critérios
CRITERIA = ['C1', 'C2', 'C3', 'C4', 'C5']
CRITERIA_NAMES = {
    'C1': 'Transparency',
    'C2': 'Computational cost (inv.)',
    'C3': 'Response time (inv.)',
    'C4': 'Technological maturity',
    'C5': 'Fault coverage'
}

print("Módulos:", MODULES)
print("Total DAs:", sum(len(v) for v in MODULE_VALUES.values()))
print("Espaço bruto:", np.prod([len(v) for v in MODULE_VALUES.values()]), "configurações")

Módulos: ['D', 'I', 'ID', 'R']
Total DAs: 14
Espaço bruto: 144 configurações


## CCA - Análise de Consistência Cruzada

### Prompts e Respostas — 4 LLMs
 
**Data:** Abril 2026   
**LLMs:** Claude Opus 4.7 · Gemini 2.0 Flash · GPT-5.3 · DeepSeek-R1

 
#### Prompt de Contexto (Documento 1)

~~~
================================================================================
CONTEXT — FDIIR DOMAIN
================================================================================
 
You are a technical expert in fault detection, isolation, identification,
and recovery (FDIIR) systems for autonomous vehicles with embedded hardware
constraints (ROS 2 / Autoware Universe, non-intensive processing, hard
real-time requirements).
 
The following descriptions define 14 design alternatives (DAs) organized
in four modules: Detection (D), Isolation (I), Identification (ID), and
Recovery (R). Read them carefully.
 
After this message I will send two evaluation prompts. Your responses must
be based on the technical content provided in this context.
 
Acknowledge receipt of this context before I proceed.
 
================================================================================
FIELD STRUCTURE
================================================================================
 
D — Detection Method:
  D1 = model-based     (Kalman filter, observers, residual analysis)
  D2 = signal-based    (statistical analysis of raw signals, threshold, wavelet)
  D3 = data-driven     (supervised/unsupervised ML, deep learning)
  D4 = active diagnosis (signal injection to provoke detectable system response)
 
I — Isolation Method:
  I1 = output comparison  (compares sensor against reference model or other sensors)
  I2 = voting             (majority decision, N-version, Byzantine-tolerant protocols)
  I3 = dependency modeling (Bayesian networks, Petri nets — maps fault propagation)
  I4 = adaptive filters   (learns nominal behavior of each sensor individually)
 
ID — Identification Method:
  ID1 = rule-based    (explicit logical conditions, expert systems)
  ID2 = statistical   (distribution fitting, hypothesis testing, PCA)
  ID3 = ML            (supervised classifier operating on already-localized fault)
 
R — Recovery Method:
  R1 = exclusion           (removes faulty sensor from fusion pipeline)
  R2 = weight reconfiguration (redistributes sensor contributions in fusion)
  R3 = bypass              (physical alternative route via redundant hardware)
 
================================================================================
DETECTION METHODS
================================================================================
 
--------------------------------------------------------------------------------
[D1 — model-based detection]
--------------------------------------------------------------------------------
 
  How it works:
    Utilizes mathematical observers (Kalman Filters, Extended Kalman Filters,
    zonotopic LPV-EKF) to estimate system states and generate residuals — the
    difference between predicted and observed measurements. The state transition
    matrix F incorporates ego-motion correction:
      F = [cos(Δψ_k)   sin(Δψ_k)]
          [-sin(Δψ_k)  cos(Δψ_k)]
    where Δψ_k is the yaw orientation change. The "Conflict factor K" from
    belief function theory measures inter-sensor disagreement. Detection is
    finalized via Pignistic Probability transformation over binary hypotheses
    Ω = {∃, ¬∃}. Zonotopic implementations are robust to bounded non-Gaussian
    disturbances, keeping state estimates within a verified envelope.
    Source: Grubmüller (III-A); Bader et al. (2017b); Conejo et al. (2025)
 
  Required inputs:
    Sensor input/output signals (e.g., GNSS position, speedometer, raw
    measurements z_k), high-fidelity dynamic models, analytical redundancy.
 
  Produced outputs:
    State estimates x̂_k, innovation matrices P_k, residual signals, agreement
    indicators via soft threshold functions, Conflict factor K.
 
  Architectural assumptions:
    Gaussian noise r_k ~ N(0, R_k); single-fault assumption for computational
    tractability; availability of accurate ground-truth models for nominal
    sensor performance.
    Source: Grubmüller; Goelles (Section 3)
 
  Known limitations:
    Performance strictly bound to mathematical model fidelity. Incorrect KF
    model or mischaracterized noise triggers active track surges (e.g., 30
    tracks for 2 objects). Struggles with high-dimensional datasets where
    distributions cannot be characterized. Brittle under SOTIF conditions
    (highly dynamic urban environments, unknown aleatory variables).
    Source: Erhan (3.1, 3.2); Gao et al.
 
  Documented interactions:
    - [R2] | Architectural integration | Analytical redundancy combined with
      weighted average voting systems to monitor sensor and software faults.
      | Source: Hamadi et al. (2022)
    - [R3] | Dependency (trigger) | D1 provides the primary residual trigger
      for R3 KF bank switching, enabling instantaneous hardware branch masking.
      | Source: Grubmüller
    - [I1] | Synergy | Residual outputs serve as the primary baseline for
      comparison logic in output comparison. | Source: Doc 15
    - [D3/ID3] | Synergy | Model-based residuals serve as high-signal features
      for downstream classifiers. | Source: Gao et al.
    - [I4] | Synergy | Residual outputs provide baseline for adaptive filtering.
      | Source: Doc 13
 
--------------------------------------------------------------------------------
[D2 — signal-based detection]
--------------------------------------------------------------------------------
 
  How it works:
    Monitors raw signals for anomalies using frequency-domain and time-domain
    techniques. Methods include: Median Filtering kernel f(l,m) = med{g(l-u,
    m-v)} for non-linear impulse noise suppression; Discrete Wavelet Transform
    (DWT) for transient feature capture; Power Spectral Density (PSD) analysis;
    Mahalanobis Distance (MD) as statistical gating:
      d_{i,k} = sqrt(ν_{i,k}^T (CP_{k|k-1}C^T + R_k)^{-1} ν_{i,k})
    Measurements exceeding threshold γ (χ² distribution) are discarded as
    outliers. Euclidean distance between sensor observation points:
      τ = sqrt((x1-x2)² + (y1-y2)²)
    Detection threshold defined by ratio τ/W, where W = 1.6m.
    Source: Elhoseny (4.1.1); Grubmüller (3.1); Hou et al. (2023)
 
  Required inputs:
    Raw sensor data streams (spin rates, pulse intensities, Bayer-level pixel
    distributions, radar point clouds, camera center points).
 
  Produced outputs:
    Signal health metrics, wavelet coefficients, frequency-domain deviation
    scores, Mahalanobis distance values, anomaly flags.
 
  Architectural assumptions:
    Quasi-stationarity of noise processes; low-level hardware interface access
    to sensor front-end; sensors providing continuous signal streams or
    overlapping fields of view.
    Source: Erhan (3.3); Hao et al. (2019)
 
  Known limitations:
    Brittle in messy environmental conditions where signal noise overlaps with
    legitimate detection thresholds. Heavily dependent on fixed thresholds —
    susceptible to environment-induced noise misidentified as fault. Ignores
    dynamic system inputs.
    Source: Koopman & Wagner (2017); Goelles et al. (2020); Gao et al.
 
  Documented interactions:
    - [I1] | Dependency | Frequently linked to output comparison for
      cross-sensor validation and localizing hardware faults. | Source: Doc 15
    - [D3] | Combination | Frequently coupled with kNN classifiers to isolate
      specific mechanical failures. | Source: Gao et al.
 
--------------------------------------------------------------------------------
[D3 — data-driven detection]
--------------------------------------------------------------------------------
 
  How it works:
    Employs deep learning architectures (CNNs, LSTMs, Autoencoders, GANs) to
    learn features automatically from large datasets. Anomalies identified when
    reconstruction error exceeds trained threshold. Enhanced RPN + FEN using
    ResNet backbone with composite weighted loss:
      q = (q_conf·w1 + q_bbox·w2 + q_cls·w3) / 3
    GAN-based OOD detection uses KL divergence and BCE loss to identify
    anomalous samples outside decision boundary. Post-hoc statistics (Cosine
    Similarity, Mahalanobis) over bottleneck layer for OOD quantification.
    Denoising Shrinkage Autoencoders (DSAE) with shrinkage block using soft
    thresholding to filter near-zero features, enhancing anomaly representation.
    Source: Elhoseny (Lee et al., 2024); Nitsch et al.; Erhan; Min et al. (2023)
 
  Required inputs:
    Large-scale historical datasets (labeled or unlabeled); video/image input;
    multi-sensor observation data; GPU/FPGA acceleration.
 
  Produced outputs:
    Anomaly scores, feature-space reconstruction errors, fault clusters,
    confidence measures, OOD flags.
 
  Architectural assumptions:
    Anomalies statistically rare and distinct from training data; requires
    specialized acceleration hardware (FPGAs/GPUs); Edge devices often
    constrained to ~20kB memory.
    Source: Erhan (4.3, 6.2)
 
  Known limitations:
    Black box nature — lacks formal verification and requirements-to-test
    traceability (ISO 26262 V-model barrier). Overconfidence of softmax in
    standard NNs for OOD detection. Class imbalance: fault data rare vs.
    normal data. High time complexity burdens Edge devices. Distributional
    shifts degrade performance.
    Source: Koopman & Wagner (2017); Nitsch et al.; Erhan (4.5, 5.3)
 
  CONFLICT — D3 certification:
    Elhoseny reports 97.78% accuracy; Koopman warns of inscrutability and
    formal guarantee absence. Both positions are valid in different contexts
    (performance vs. certification).
 
  Documented interactions:
    - [Hardware/Edge] | Dependency | Deep dependency on high-performance
      specialized hardware (FPGAs/GPUs) at the Edge/Fog layer.
      | Source: Erhan (Section 5)
    - [D1] | Hybrid framework | CNN-Kalman Filter hybrid: CNNs handle feature
      extraction for model-based filters. | Source: van Wyk et al. (2020)
    - [R3] | Recovery trigger | GAN assigns low confidence to OOD samples,
      flagging them for recovery intervention. | Source: Nitsch et al.
    - [ID3] | Combination | Paired with PCA for feature extraction to reduce
      computational overhead. | Source: Gao et al.
 
--------------------------------------------------------------------------------
[D4 — active diagnosis]
--------------------------------------------------------------------------------
 
  How it works:
    Proactively injects test signals to provoke detectable system responses,
    distinguishing faulty modes from normal operation. Classified as stochastic
    (affecting probability ratio tests) or deterministic (minimum energy signals
    for model identification). In Lidar: adjusting pulse timing or modulation
    to distinguish legitimate echoes from jamming/crosstalk. In motors: current
    pulse injection into d-axis current for winding resistance estimation.
    Source: Gao et al. (Section III-B); Goelles (6.3)
 
  Required inputs:
    Actively modified transmitter commands; bi-directional control of sensor
    transmitter hardware.
 
  Produced outputs:
    Verified return signals vs. discarded noise/interference; fault mode
    discrimination.
 
  Architectural assumptions:
    Hardware capability for variable pulse modulation; bi-directional sensor
    control access.
 
  Known limitations:
    May temporarily reduce effective sampling rates or increase processing
    overhead. Mandate to minimize adverse effects on real-time performance
    and passenger comfort during test interval.
    Source: Gao et al. (Section III-B)
 
  CONFLICT — D4 operational viability:
    Goelles (6.3): viable in operation for anti-DoS probing.
    Gao et al.: characterized as far from mature (2015); more recent work
    (Sinha et al., 2023) evolved toward Runtime Monitors as system-level
    diagnostic intervention.
    Motor domain example (Gao et al.) is non-AV context.
 
  Documented interactions:
    - [R2] | Synergy (verification) | D4 can be used to verify if weight
      reconfiguration successfully restored sensor functionality by probing
      the sensor post-recovery. | Source: Goelles (6.3)
 
================================================================================
ISOLATION METHODS
================================================================================
 
--------------------------------------------------------------------------------
[I1 — output comparison]
--------------------------------------------------------------------------------
 
  How it works:
    Compares primary sensor outputs against reference model or secondary sensor
    of different modality. Isolation via innovation checking using Mahalanobis
    Distance:
      d_{i,k} = sqrt(ν^T_{i,k} P̂^{-1}_{k|k-1} ν_{k,i})
    where ν_{i,k} = z_{i,k} - Cx̂_{k|k-1} is the innovation. If d_{i,k}
    exceeds χ² threshold, measurement isolated as discrepancy. Kuhn-Munkres
    bipartite graph matching enforces 1:1 association between tracker subset
    and detections, identifying orphaned detections or ghost tracks as fault
    triggers. Distinguishes hardware fault from software fault.
    Source: Grubmüller (III-A); Lee (III-A); Kaci et al. (2014)
 
  Required inputs:
    Synchronized outputs from multiple perception sources; redundant sensor
    block outputs; fusion residues. Requires overlapping fields of view.
 
  Produced outputs:
    Conflict vectors identifying non-conforming sensor node; fault localization
    flag (Hardware vs. Software).
 
  Architectural assumptions:
    Hardware and/or software analytical redundancy; precise extrinsic
    calibration; robust sensor fusion layer.
    Source: Goelles (6.2)
 
  Known limitations:
    Only as robust as the reference sensor — if reference drifts or shares
    common-mode software failure, isolation fails. High complexity in cross-
    modal data transformation. Vulnerable to common-cause failures.
    Source: Grubmüller; Goelles (6.2)
 
  Documented interactions:
    - [D1] | Dependency | Heavily dependent on D1 for generation of residuals.
      | Source: Doc 15
    - [ID1] | Dependency | Interaction with joint calibration for coordinate
      transformation. | Source: Doc 15
    - [I4/D1] | Synergy | Residual outputs from D1 provide baseline for
      comparison and adaptive filtering. | Source: Doc 13
 
--------------------------------------------------------------------------------
[I2 — voting]
--------------------------------------------------------------------------------
 
  How it works:
    Multi-sensor majority consensus using N-version programming and N
    self-checking programming. Byzantine fault tolerance requires:
      n >= 3f + 1
    where n = total modules, f = tolerable arbitrary faults. Soft Voting
    Ensemble averages class probabilities from black-box and fusion-based
    classifiers. Bayesian Networks model dependencies to account for
    common-cause failures. Heterogeneous redundancy: Camera, LiDAR, Radar
    cross-validation using Mahalanobis distance thresholds.
    Source: Grubmüller (Table I); Lee (III-D); Mendonça et al. (2023)
 
  Required inputs:
    Heterogeneous sensor data streams; prior probability distributions of
    failure modes; independent module outputs.
 
  Produced outputs:
    Consensus-based state estimate; majority-consensus health status;
    localized fault identification; reliability weights.
 
  Architectural assumptions:
    Majority of sensing population remains uncompromised; heterogeneous
    redundancy (physical hardware required).
    Source: Goelles (1)
 
  Known limitations:
    Vulnerable to common-cause failures (e.g., regional heavy fog). High
    hardware cost, physical space, and energy consumption. Non-rejuvenating
    systems: number of tolerable faults strictly capped by total module count.
 
  CONFLICT — Co-fault vulnerability:
    Tian et al. identify that strong light source can simultaneously overexpose
    camera AND saturate LiDAR, causing total failure of voting logic.
    Source: Tian et al.
 
  Documented interactions:
    - [R1] | Combination | Frequently combined with R1 for identifying which
      Byzantine module to exclude. | Source: Mendonça et al. (2023)
    - [Hardware] | Dependency | Cannot be implemented as software-only;
      requires physical hardware redundancy. | Source: Doc 13
 
--------------------------------------------------------------------------------
[I3 — dependency modeling]
--------------------------------------------------------------------------------
 
  How it works:
    Maps fault propagation using Coloured Probabilistic Time Petri Nets (CPTPN),
    Bayesian Networks, and Signed Directed Graphs (SDG). Bayesian Networks use
    Directed Acyclic Graphs of sensor dependencies to generate random residuals
    representing failure probabilities. Enables reachability analysis: formally
    verifying the system reaches a safe degraded state within mandated time.
    Source: Emzivat et al. (2018); Erhan (4.3); Gao et al.
 
  Required inputs:
    Probabilistic relationships between hardware nodes; historical fault
    signatures; system topological data; system state probabilities.
    System topology must be well-defined and stationary.
 
  Produced outputs:
    Probability distribution of fault origin; failure path localization;
    risk assessments; reachability verification of safe states.
 
  Architectural assumptions:
    Comprehensively modeled sensor interdependencies; well-defined stationary
    system topology; prior knowledge of causal architecture.
    Source: Erhan (4.3)
 
  Known limitations:
    High computational cost for large sensor counts. Difficulty distinguishing
    between true anomalies and environmental contamination (dirt, mud).
    Exhaustive modeling of every failure path is a primary bottleneck.
    Source: Erhan (2.1); Doc 13
 
  Documented interactions:
    - [D3/ID3] | Dependency | Necessitates combination with Recurrent NNs
      for temporal fault signature analysis. | Source: Gao et al.
    - [R3] | Direct link | I3 fault localization output directly informs
      the recovery module. | Source: Doc 15
 
--------------------------------------------------------------------------------
[I4 — adaptive filters]
--------------------------------------------------------------------------------
 
  How it works:
    Employs unknown input observers and adaptive Kalman filters (including
    Unscented Kalman Filter for non-linear dynamics) to learn nominal behavior
    of individual sensors. Weakening factors p_trust in [0,1] discount belief
    masses of suspected faulty sources before global combination. Channel-
    specific training decouples input disturbances from actual hardware faults.
    NOTE: warm-up period required on initialization before baseline is stable.
    Source: Bader (II); Goelles (6.2); Erhan (4.6); van Wyk et al. (2020)
 
  Required inputs:
    Continuous time-series measurements per sensor channel; non-stationary
    data streams for ongoing model adaptation.
 
  Produced outputs:
    Dynamic health indicators; sensor-specific deviation thresholds; decoupled
    input disturbances; validated sensor readings.
 
  Architectural assumptions:
    Non-stationary data streams required to maintain model accuracy. Designed
    to detect deviations from learned nominal behavior.
    Source: Erhan (2.1); Goelles (6.2)
 
  Known limitations:
    Extremely sensitive to initial parameter tuning and warm-up periods.
    Requires stable periods for initial behavior learning. Traditional
    stationary algorithms fail as environmental distributions evolve.
    Source: Goelles (6.2); Erhan (2.1)
 
  Documented interactions:
    - [D3] | Synergy | Synergy with D3 for deep learning-based feature
      extraction. | Source: van Wyk et al. (2020)
    - [D1/I1] | Synergy | Residual outputs from D1 provide baseline for
      adaptive filtering. | Source: Doc 13
 
================================================================================
IDENTIFICATION METHODS
================================================================================
 
--------------------------------------------------------------------------------
[ID1 — rule-based identification]
--------------------------------------------------------------------------------
 
  How it works:
    Uses expert-defined if-then logical conditions and boolean gates to classify
    fault signatures and categorize faults into predefined taxonomies.
    Four formal conditions for dual-sensor branch (Bader, 2017):
      1. Agreement (Existence): BetP(exists) >= th -> no fault
      2. Agreement (Absence): BetP(exists) <= 1-th -> no fault
      3. Discrepancy (Fault): {p_P=1, p_C=0} -> 1-th < BetP(exists) < th -> FAULT
      4. Discrepancy (Fault): {p_P=0, p_C=1} -> 1-th < BetP(exists) < th -> FAULT
    where th > 0.5. Taxonomy of 32 distinct camera failure modes documented.
    Source: Bader (2017); Ceccarelli and Secci (2023); Gao et al.
 
  Required inputs:
    Detected residuals; pre-defined fault pattern libraries; visual signatures;
    coordinate mapping; self-diagnosis confirmation.
 
  Produced outputs:
    Categorical fault type; estimated severity level; specific fault category
    (e.g., active vs. passive faults).
 
  Architectural assumptions:
    Complete taxonomy of all expected failure modes must be pre-defined;
    effective only for known, documented fault signatures.
    Source: Goelles (6.2); Erhan (3.3)
 
  Known limitations:
    Poor performance against novel environmental anomalies. Low generality
    and low expandability — highly system-specific.
    Source: Erhan (3.3); Gao et al.
 
  Documented interactions:
    - [I1] | Dependency | Interaction with output comparison for coordinate
      transformation. | Source: Doc 15
    - [R3] | Dependency | Categorical fault type from ID1 can trigger targeted
      bypass of the identified faulty sensor. | Source: Doc 15
 
--------------------------------------------------------------------------------
[ID2 — statistical identification]
--------------------------------------------------------------------------------
 
  How it works:
    Uses PCA, ICA, hypothesis testing, and Mahalanobis Distance to quantify
    deviation from ground truth distribution and identify fault type.
    Mahalanobis Distance identifies statistical independence between LM tracks.
    Conflict factor K from Dempster's rule acts as direct statistical indicator
    of source defectiveness. PCA projects complex data into lower-dimensional
    subspace for fault separability. ICA allows non-Gaussian latent variables
    for non-standard noise pattern identification. Distinguishes fault
    manifestation: gradual drift vs. instant anomaly using Gaussian variable
    scaling.
    Source: Grubmüller (III-A); Bader; Erhan (3.1, 3.4); van Wyk et al. (2020)
 
  Required inputs:
    Multidimensional sensor data streams; time-series sensor data; historical
    performance data; self-diagnosis confirmation; observation vectors.
 
  Produced outputs:
    Quantifiable deviation metrics; dimensionality-reduced fault vectors;
    anomaly type and severity level.
 
  Architectural assumptions:
    Data can be projected into lower-dimensional subspace where faults are
    separable; sensors with self-diagnosis capabilities.
    Source: Erhan (3.6)
 
  Known limitations:
    High computational overhead; limited interpretability for non-experts.
    Standard SVMs struggle with highly transitional dynamics. Relies on
    specific distribution assumptions.
    Source: Erhan; Gao et al.; Min et al. (2023)
 
  Documented interactions:
    No explicit interactions documented in source context.
 
--------------------------------------------------------------------------------
[ID3 — ML identification]
--------------------------------------------------------------------------------
 
  How it works:
    Supervised classifiers (SVM, NN, RPN) classify localized fault types.
    Includes explicit Fault Class in output set: if fusion-based enhancer
    confidence is low -> object reclassified as Fault to invoke recovery.
    CCBMO optimization tunes RPN weights to increase precision of Fault Class
    detections. Incremental learning allows adaptation to new error observations
    without full retraining.
    Source: Lee (III-C); Elhoseny (6.2); Gao et al.
 
  Required inputs:
    Localized fault data; labeled training signatures; multi-modal sensor
    regions; object detection loss, bounding box loss, classification loss.
 
  Produced outputs:
    Fault classification; confidence scores; characterized anomaly type.
 
  Architectural assumptions:
    Relevance between labeled training data and real-world distributions;
    GPU/FPGA acceleration required; high-fidelity training data per anomaly type.
    Source: Goelles (7.5)
 
  Known limitations:
    Limited by black box commercial sensors restricting raw data access.
    High dependence on annotated historical data. Cannot identify faults
    not encountered during training.
    Source: Goelles (7.5)
 
  Documented interactions:
    - [D3] | Dependency | Dependency on D3 foundations; frequently paired
      with PCA for feature extraction. | Source: Doc 15; Gao et al.
    - [SOTIF] | Synergy | Waypoint prediction accuracy serves as direct metric
      for SOTIF (ISO PAS 21448) adherence. | Source: Secci via Doc 13
 
================================================================================
RECOVERY METHODS
================================================================================
 
SCOPE NOTE — R2:
  Consolidated as weight redistribution and discounting only.
 
SCOPE NOTE — R3:
  Operates exclusively within the perception system. Fallback-Safe MPC,
  Safing Mission, Landmark Odometry and Minimal Risk Maneuver references
  excluded — all involve planning-level decisions outside perception scope.
 
NOTE ON R1:
  Software rejuvenation excluded from R1 — functionally closer to R3
  (bypass via alternative software path).
 
--------------------------------------------------------------------------------
[R1 — exclusion]
--------------------------------------------------------------------------------
 
  How it works:
    Logical removal of faulty sensor or branch from global fusion pipeline.
    KF Bank and Switch mechanism excludes isolated noisy sensor from global
    fusion algorithm. Highly distorted radar measurements immediately excluded
    to prevent track explosion. Architecture relies on remaining healthy
    redundant sensors for continued operation.
    Source: Grubmüller (IV-A); Goelles (1, 6.3)
 
  Required inputs:
    Fault location from Isolation; redundant healthy sensor data; fault
    isolation and identification status.
 
  Produced outputs:
    Reconfigured sensor fusion list omitting faulty nodes; filtered fused
    state estimate.
 
  Architectural assumptions:
    Sufficient sensor redundancy to maintain safety after node exclusion;
    high-level fusion manager with dynamic exclusion logic.
    Source: Goelles
 
  Known limitations:
    Sudden loss of FOV or modality increases risk in complex scenes. System
    performance degrades with each excluded component. Reduces system
    redundancy immediately. Strong light interference can simultaneously blind
    Camera and LiDAR, rendering exclusion ineffective.
    Source: Goelles (1); Tian et al.
 
  Documented interactions:
    - [I2] | Combination | Direct link to voting for identifying which
      Byzantine module to exclude. | Source: Mendonça et al. (2023)
 
--------------------------------------------------------------------------------
[R2 — weight reconfiguration]
--------------------------------------------------------------------------------
 
  How it works:
    Dynamically adjusts sensor influence in fusion without full exclusion.
    Discounting Factor weakens belief masses of suspect sources before global
    combination. Kalman Filters evaluate sensor accuracy in real-time, adjusting
    Dynamic Weight based on recent innovation residuals. Weakening factors
    p_trust in [0,1] ensure faulty hardware outputs masked without severing
    contribution. Enables fail-operational behavior. Environmental suitability
    weighting: e.g., favoring Radar over Lidar in snow.
    Source: Bader (II); Elhoseny; Goelles (4, 6.3)
 
  Required inputs:
    Sensor reliability and performance metrics; sensor health indicators;
    recent innovation residuals.
 
  Produced outputs:
    Dynamic weight vectors for fusion layer; weighted fusion result favoring
    healthy sensors.
 
  Architectural assumptions:
    Fusion algorithms capable of handling variable-weight inputs; adaptive
    sensor fusion architecture.
    Source: Goelles (4)
 
  Known limitations:
    Risk of cascading corruption if weights not reduced quickly enough.
    Corrupted data may influence fusion output before discounting takes effect.
    Source: Goelles; Sinha et al. (2023)
 
  Documented interactions:
    - [D1/I1] | Dependency | Dependency on D1/I1 for providing residue data
      needed to calculate weakening factors. | Source: Bader et al.
    - [D4] | Synergy (verification) | D4 used to verify if weight
      reconfiguration successfully restored functionality. | Source: Goelles (6.3)
 
--------------------------------------------------------------------------------
[R3 — bypass]
--------------------------------------------------------------------------------
 
  How it works:
    Redirects data flow around a failed sensor or processing module using an
    alternative route within the perception pipeline. Primary mechanism:
    KF Bank and Switch — switches the global fusion input from the faulty
    sensor branch to a redundant fault-free sensor branch, masking the hardware
    fault without disrupting the perception pipeline. Operates exclusively
    within the perception system.
    Source: Grubmüller; Goelles (6.3)
 
  Required inputs:
    Fault localization signal from Isolation module; availability of redundant
    sensor branch or alternative data path within the perception pipeline.
 
  Produced outputs:
    Reconfigured perception pipeline routing data around the failed component;
    continued perception output from healthy redundant path.
 
  Architectural assumptions:
    Existence of a redundant sensor branch or alternative data path within the
    perception pipeline architecture; fault has been localized by Isolation.
    Source: Grubmüller; Goelles
 
  Known limitations:
    Requires redundant hardware or alternative data path — not always present.
    Bypass reduces diversity of fusion input. Validating bypass routing across
    all fault scenarios requires extensive testing.
    Source: Goelles (4, 6.3)
 
  Documented interactions:
    - [D1] | Dependency | D1 residuals as primary trigger for KF bank switching.
      | Source: Grubmüller
~~~

#### Prompt A — Avaliacao do Vetor A

~~~
================================================================================
EVALUATION PROMPT A — VECTOR A (DA QUALITY SCORES)
================================================================================
 
IDENTIFICATION:
Before providing any assessment, identify yourself on the first line:
  LLM: [your model name and version]
Then proceed with the full evaluation.
 
CONTEXT:
Based on the technical context provided, evaluate each of the 14 design
alternatives (DAs) across 5 criteria. Each DA represents a method choice
within one module of an FDIIR pipeline for autonomous vehicle perception.
 
SYSTEM CONSTRAINTS (fixed for all assessments):
  - Pipeline: Detection -> Isolation -> Identification -> Recovery
  - Platform: ROS 2 / Autoware Universe, local embedded processing
  - Hard real-time constraints apply
  - Fault scope: abrupt faults only (data loss, Gaussian noise, dead zone,
    intensity corruption) — gradual/incipient faults are OUT OF SCOPE
  - System scope: all modules operate exclusively within the perception
    system. Planning-level decisions are OUT OF SCOPE
 
SCALE:
  1 = very poor
  2 = poor
  3 = good
  4 = excellent
 
CRITERIA:
  C1 Transparency: degree to which the internal decision process is
     visible and documentable, enabling auditability and certifiability
     (ISO 26262)
     Scale: 1-4 (4 = maximum transparency)
  C2 Computational cost (inverted): CPU/memory resources at runtime
     Scale: 1-4 (4 = lightest)
  C3 Response time (inverted): time to produce output after receiving
     input
     Scale: 1-4 (4 = fastest response)
  C4 Technological maturity: degree of validation in the autonomous
     vehicle literature
     Scale: 1-4 (4 = maximum maturity)
  C5 Fault coverage: adequacy for the abrupt fault set defined in scope
     (data loss, Gaussian noise, dead zone, intensity corruption) in a
     heterogeneous sensor suite (camera + LiDAR + Radar)
     Scale: 1-4 (4 = maximum coverage)
 
INSTRUCTIONS:
  1. Evaluate each DA on each criterion independently.
  2. Assign exactly one value from {1, 2, 3, 4} per cell.
  3. Provide a one-sentence justification per cell.
  4. Do not reference previous evaluations or existing scores.
     Evaluate from first principles based on the context provided.
 
RESPONSE FORMAT (repeat for all 14 DAs in order:
D1, D2, D3, D4, I1, I2, I3, I4, ID1, ID2, ID3, R1, R2, R3):
 
  [D1 — model-based detection]
    C1 Transparency        : X | reason=...
    C2 Computational cost  : X | reason=...
    C3 Response time       : X | reason=...
    C4 Maturity            : X | reason=...
    C5 Fault coverage      : X | reason=...
================================================================================
~~~

#### Respostas — Prompt A

##### Claude Opus 4.7

LLM: Claude Opus 4.7 (Anthropic)
 
[D1 — model-based detection]
  C1 Transparency        : 4 | reason=Explicit state-space equations, residuals, and observer dynamics are fully documentable and auditable against ISO 26262 V-model requirements.
  C2 Computational cost  : 3 | reason=Recursive KF/EKF updates involve moderate matrix operations but are well-optimized for embedded ROS 2 deployment.
  C3 Response time       : 4 | reason=Single-step recursive residual generation per cycle yields deterministic low-latency output suitable for hard real-time.
  C4 Maturity            : 4 | reason=Extensively validated in the AV literature (Grubmüller, Bader, Conejo) and a baseline for the other modules' interactions.
  C5 Fault coverage      : 3 | reason=Residual growth flags data loss and Gaussian noise well in heterogeneous suites, but brittleness to model mismatch and SOTIF conditions limits robustness.
 
[D2 — signal-based detection]
  C1 Transparency        : 4 | reason=Thresholds, wavelet coefficients, and Mahalanobis gating are explicit numerical tests with traceable decision logic.
  C2 Computational cost  : 3 | reason=Median filters and thresholding are lightweight; DWT/PSD add moderate cost but fit embedded budgets.
  C3 Response time       : 4 | reason=Per-sample or per-window signal checks produce near-instantaneous flags with minimal buffering.
  C4 Maturity            : 3 | reason=Classical signal processing methods well-established across Elhoseny, Grubmüller, and Hou et al., though less AV-specific than D1.
  C5 Fault coverage      : 3 | reason=Strong fit for intensity corruption, dead zones, and impulse noise, but fixed thresholds can confuse environmental noise with genuine faults.
 
[D3 — data-driven detection]
  C1 Transparency        : 1 | reason=Explicitly described as black box with ISO 26262 V-model traceability barrier and no formal verification path.
  C2 Computational cost  : 1 | reason=Deep architectures (CNN/LSTM/GAN) demand GPU/FPGA acceleration that conflicts with non-intensive embedded processing constraints.
  C3 Response time       : 2 | reason=Inference latency and high time complexity burden Edge devices, jeopardizing hard real-time determinism.
  C4 Maturity            : 2 | reason=Strong reported accuracy (Elhoseny 97.78%) but certification gaps and distributional shift vulnerabilities limit validated AV maturity.
  C5 Fault coverage      : 3 | reason=Autoencoder reconstruction error and OOD detection cover varied abrupt anomalies, though rare-class imbalance can leave gaps.
 
[D4 — active diagnosis]
  C1 Transparency        : 3 | reason=Deterministic injection and probability ratio tests are interpretable, but stochastic variants reduce clarity of per-cycle decisions.
  C2 Computational cost  : 3 | reason=Pulse modulation and response checks add modest overhead beyond the baseline signal chain.
  C3 Response time       : 2 | reason=Test intervals and response-settling time introduce latency that conflicts with hard real-time mandates.
  C4 Maturity            : 2 | reason=Gao et al. characterize it as far from mature and AV evidence is thin; Goelles supports viability only for anti-DoS probing.
  C5 Fault coverage      : 2 | reason=Well-suited to jamming/crosstalk and active faults but poorly matched to passive data-loss and Gaussian-noise classes in scope.
 
[I1 — output comparison]
  C1 Transparency        : 4 | reason=Mahalanobis innovation checks and Kuhn-Munkres bipartite matching are mathematically explicit and inspectable.
  C2 Computational cost  : 3 | reason=Cross-modal comparison and assignment solvers are tractable but non-trivial on embedded hardware.
  C3 Response time       : 3 | reason=Per-frame comparison is fast, though cross-modal transformation adds pipeline stages.
  C4 Maturity            : 4 | reason=Canonical isolation mechanism in AV fault-diagnosis literature (Grubmüller, Lee, Kaci) with well-understood behavior.
  C5 Fault coverage      : 4 | reason=Heterogeneous camera/LiDAR/radar overlap and HW/SW fault distinction directly target the defined abrupt fault set.
 
[I2 — voting]
  C1 Transparency        : 4 | reason=Majority and Byzantine protocols (n >= 3f + 1) are formally specifiable and verifiable — classical safety-engineering primitives.
  C2 Computational cost  : 2 | reason=N-version execution multiplies compute and memory footprint, straining non-intensive embedded budgets.
  C3 Response time       : 3 | reason=Consensus resolves quickly but must synchronize and wait on all voters before producing a decision.
  C4 Maturity            : 4 | reason=Decades of safety-critical practice plus documented AV use (Mendonça et al.) establish high maturity.
  C5 Fault coverage      : 3 | reason=Heterogeneous cross-validation handles most abrupt faults, but common-cause scenarios (Tian et al. light saturation) create blind spots.
 
[I3 — dependency modeling]
  C1 Transparency        : 3 | reason=Graphs/CPTPN/SDG structures are inspectable, but probabilistic inference and large topologies reduce cell-level clarity.
  C2 Computational cost  : 2 | reason=Explicitly flagged as high computational cost for large sensor counts, taxing embedded platforms.
  C3 Response time       : 2 | reason=Probabilistic propagation and reachability checks introduce inference latency relative to direct comparison.
  C4 Maturity            : 3 | reason=Solid theoretical foundation and AV studies (Emzivat, Gao), though less ubiquitous than I1/I2.
  C5 Fault coverage      : 2 | reason=Strong for path-based propagation but documented difficulty separating true anomalies from environmental contamination narrows abrupt-fault coverage.
 
[I4 — adaptive filters]
  C1 Transparency        : 3 | reason=UKF and unknown-input observer equations are explicit, but online adaptation weights shift interpretability over time.
  C2 Computational cost  : 3 | reason=Adaptive KF/UKF variants carry moderate overhead appropriate for embedded deployment.
  C3 Response time       : 3 | reason=Recursive structure is fast once stable, but the required warm-up period delays initial availability.
  C4 Maturity            : 3 | reason=Well-established filtering family with documented AV applications (Bader, van Wyk), yet less canonical for isolation than I1.
  C5 Fault coverage      : 3 | reason=Per-channel deviation detection handles data loss and noise well, though abrupt faults during warm-up can evade baselines.
 
[ID1 — rule-based identification]
  C1 Transparency        : 4 | reason=Explicit if-then conditions with BetP thresholds are the most auditable form, directly usable in ISO 26262 traceability.
  C2 Computational cost  : 4 | reason=Boolean gate evaluation is among the cheapest operations available on embedded hardware.
  C3 Response time       : 4 | reason=Rule evaluation is O(1) per condition, delivering immediate classification.
  C4 Maturity            : 4 | reason=Bader's 32-mode camera taxonomy and Ceccarelli/Secci work demonstrate long-standing AV validation.
  C5 Fault coverage      : 3 | reason=The bounded scope fault set is well-taxonomized, though novel variants would escape pre-defined rules.
 
[ID2 — statistical identification]
  C1 Transparency        : 3 | reason=PCA loadings and hypothesis tests are interpretable, but ICA and high-dimensional distributions reduce direct auditability.
  C2 Computational cost  : 2 | reason=Explicitly noted as high computational overhead, particularly for multi-dimensional projections at runtime.
  C3 Response time       : 3 | reason=Post-training projections are fast per sample, though updating covariance models adds intermittent cost.
  C4 Maturity            : 3 | reason=PCA/ICA/Mahalanobis are classical and broadly cited in the AV FDIIR literature.
  C5 Fault coverage      : 3 | reason=Explicitly distinguishes gradual drift vs. instant anomaly, providing solid coverage of the abrupt fault classes in scope.
 
[ID3 — ML identification]
  C1 Transparency        : 1 | reason=Supervised classifiers operate as black boxes and are further obscured by commercial sensors restricting raw-data access.
  C2 Computational cost  : 1 | reason=Requires GPU/FPGA acceleration, conflicting directly with the non-intensive embedded processing constraint.
  C3 Response time       : 2 | reason=Neural inference latency is predictable but higher than rule-based or statistical alternatives on constrained hardware.
  C4 Maturity            : 2 | reason=Active research with AV demonstrations (Lee, Elhoseny) but limited certification record and heavy labeled-data dependence.
  C5 Fault coverage      : 3 | reason=Explicit Fault Class and incremental learning cover the in-scope abrupt set when labels exist, but unseen variants are missed.
 
[R1 — exclusion]
  C1 Transparency        : 4 | reason=Binary remove/keep decisions are trivially documentable and verifiable against safety cases.
  C2 Computational cost  : 4 | reason=List reconfiguration in the fusion manager is near-zero-cost.
  C3 Response time       : 4 | reason=Immediate removal on isolation signal with no iterative computation required.
  C4 Maturity            : 4 | reason=Classical fault-tolerant fusion primitive (Grubmüller KF bank, Goelles) with long validation history.
  C5 Fault coverage      : 3 | reason=Effective for any abrupt fault once isolated, but sudden FOV/modality loss degrades safety when redundancy is thin.
 
[R2 — weight reconfiguration]
  C1 Transparency        : 3 | reason=Weight vectors and discounting factors p_trust in [0,1] are inspectable, though dynamic adjustment logic adds complexity.
  C2 Computational cost  : 3 | reason=Weight updates based on innovation residuals add modest overhead to the existing fusion layer.
  C3 Response time       : 3 | reason=Weight adjustment is fast, but cascading-corruption risk before weights converge introduces effective latency concerns.
  C4 Maturity            : 3 | reason=Dempster-Shafer discounting and dynamic-weight KF fusion are established in AV literature (Bader, Elhoseny, Goelles).
  C5 Fault coverage      : 4 | reason=Graceful degradation across all defined abrupt fault types in a heterogeneous suite enables fail-operational perception without severing contributions.
 
[R3 — bypass]
  C1 Transparency        : 4 | reason=KF Bank and Switch routing is a deterministic, fully specifiable state machine over redundant branches.
  C2 Computational cost  : 4 | reason=Pure routing/switching incurs negligible runtime cost beyond the pre-existing parallel branches.
  C3 Response time       : 4 | reason=Instantaneous branch masking on trigger — documented explicitly as such in Grubmüller.
  C4 Maturity            : 3 | reason=Well-documented in the perception-layer AV literature (Grubmüller, Goelles), though narrower deployment base than R1.
  C5 Fault coverage      : 3 | reason=Covers abrupt faults cleanly when a redundant path exists, but coverage collapses where redundant hardware is absent.

##### Gemini 2.0 Flash

LLM: Gemini 2.0 Flash
 
[D1 — model-based detection]
  C1 Transparency        : 4 | reason=Mathematical observers and state-space matrices provide an auditable and formally verifiable decision path.
  C2 Computational cost  : 3 | reason=Matrix operations for Kalman Filters are moderately demanding but well within embedded CPU capabilities.
  C3 Response time       : 3 | reason=Recursive state estimation allows for output generation at every sensor time step with minimal latency.
  C4 Maturity            : 4 | reason=Extensively validated in AV literature as a primary method for residual generation and ego-motion correction.
  C5 Fault coverage      : 3 | reason=Highly effective for noise and corruption within defined models, but brittle under unknown urban aleatory variables.
 
[D2 — signal-based detection]
  C1 Transparency        : 4 | reason=Logic based on fixed statistical thresholds and wavelet coefficients is fully visible and documentable.
  C2 Computational cost  : 4 | reason=Time-domain and frequency-domain filters (e.g., Median, MD) are computationally very light for real-time streams.
  C3 Response time       : 4 | reason=Direct signal processing enables near-instantaneous anomaly detection upon packet arrival.
  C4 Maturity            : 4 | reason=Standard industry practice for low-level hardware health monitoring and outlier rejection.
  C5 Fault coverage      : 2 | reason=Susceptible to environmental noise overlaps and fails to account for dynamic system inputs.
 
[D3 — data-driven detection]
  C1 Transparency        : 1 | reason=Black-box deep learning architectures lack formal verification and requirements-to-test traceability for ISO 26262.
  C2 Computational cost  : 1 | reason=Inference for CNNs or GANs typically requires specialized GPU/FPGA acceleration, exceeding 20kB memory constraints.
  C3 Response time       : 2 | reason=Complex feature extraction and reconstruction error calculations introduce significant processing bottlenecks.
  C4 Maturity            : 3 | reason=Highly researched for OOD detection, though industrial adoption is hindered by certification barriers.
  C5 Fault coverage      : 4 | reason=Excellent at identifying complex intensity corruptions and OOD samples that elude mathematical models.
 
[D4 — active diagnosis]
  C1 Transparency        : 3 | reason=The command-response logic for test signal injection is clear, though stochastic effects can be complex.
  C2 Computational cost  : 3 | reason=Requires bi-directional hardware control and moderate processing to analyze return signals.
  C3 Response time       : 2 | reason=Testing intervals and pulse modulation can temporarily reduce effective sensor sampling rates.
  C4 Maturity            : 2 | reason=While effective in motor domains, it is characterized as less mature for complex AV perception sensors.
  C5 Fault coverage      : 3 | reason=Strong capability to distinguish legitimate echoes from external jamming or crosstalk.
 
[I1 — output comparison]
  C1 Transparency        : 4 | reason=Discrepancy logic using Mahalanobis distance and Kuhn-Munkres matching is deterministic and clear.
  C2 Computational cost  : 3 | reason=Demands synchronized data and coordinate transformations between multiple modalities or reference models.
  C3 Response time       : 3 | reason=Latency is tied to the slowest sensor in the comparison set and the association logic overhead.
  C4 Maturity            : 4 | reason=Widely documented as a fundamental method for isolating hardware vs. software perception faults.
  C5 Fault coverage      : 3 | reason=Effective at isolating discrepancies across heterogeneous suites but vulnerable to common-cause sensor drifts.
 
[I2 — voting]
  C1 Transparency        : 4 | reason=Byzantine fault tolerance and majority consensus protocols are mathematically transparent and auditable.
  C2 Computational cost  : 4 | reason=Majority decision logic is computationally trivial, though it assumes the presence of physical hardware replicas.
  C3 Response time       : 4 | reason=Consensus checking across pre-processed sensor outputs is extremely fast for real-time pipelines.
  C4 Maturity            : 4 | reason=A cornerstone of safety-critical system design with established bounds on tolerable arbitrary faults.
  C5 Fault coverage      : 3 | reason=Excellent for abrupt hardware loss, but coverage fails if environmental conditions (e.g., light) blind multiple modalities.
 
[I3 — dependency modeling]
  C1 Transparency        : 3 | reason=Bayesian and Petri Net graphs are visible but become difficult to document as sensor counts scale.
  C2 Computational cost  : 2 | reason=Large-scale reachability analysis and probability mapping are computationally intensive for embedded systems.
  C3 Response time       : 2 | reason=The time required to traverse complex dependency graphs can delay fault localization.
  C4 Maturity            : 3 | reason=Well-established in theoretical formal verification and system-level diagnostic research.
  C5 Fault coverage      : 3 | reason=Superior for mapping fault propagation, though it struggles with environmental contamination signatures.
 
[I4 — adaptive filters]
  C1 Transparency        : 3 | reason=Internal parameter updates are visible, but the learning process introduces non-deterministic behavior during adaptation.
  C2 Computational cost  : 2 | reason=Learning nominal behavior for every individual channel concurrently places a heavy burden on runtime resources.
  C3 Response time       : 2 | reason=System performance is hampered by mandatory warm-up periods and the lag of the adaptation loop.
  C4 Maturity            : 3 | reason=Growing presence in literature for dealing with non-stationary distributions and evolving environments.
  C5 Fault coverage      : 4 | reason=Strongest at decoupling input disturbances from hardware-specific faults through individual sensor training.
 
[ID1 — rule-based identification]
  C1 Transparency        : 4 | reason=Expert-defined IF-THEN conditions and boolean gates provide the highest level of auditability.
  C2 Computational cost  : 4 | reason=Evaluating logical gates and predefined taxonomies is computationally negligible for embedded hardware.
  C3 Response time       : 4 | reason=Boolean classification logic executes near-instantaneously once residuals are available.
  C4 Maturity            : 4 | reason=Mature application in diagnostic expert systems with well-documented failure mode taxonomies.
  C5 Fault coverage      : 2 | reason=Limited strictly to known, documented fault signatures and performs poorly against novel anomalies.
 
[ID2 — statistical identification]
  C1 Transparency        : 3 | reason=Statistical metrics (PCA, MD) are auditable but require domain expertise to interpret correctly.
  C2 Computational cost  : 2 | reason=Dimensionality reduction techniques like PCA/ICA are resource-intensive when applied to high-dimensional sensor data.
  C3 Response time       : 3 | reason=Reasonably fast, though inhibited by the need for observation vector transformation.
  C4 Maturity            : 3 | reason=Validated for identifying standard noise patterns, yet relies heavily on specific distribution assumptions.
  C5 Fault coverage      : 3 | reason=Effective at quantifying the severity and manifestation of faults like gradual drift vs. abrupt anomalies.
 
[ID3 — ML identification]
  C1 Transparency        : 1 | reason=Supervised classifiers function as black boxes, making them difficult to certify for high-integrity AV systems.
  C2 Computational cost  : 1 | reason=Supervised identification of localized faults requires substantial GPU/FPGA acceleration.
  C3 Response time       : 2 | reason=The depth of the classification architecture determines the latency between localization and identification.
  C4 Maturity            : 3 | reason=Highly active research area, particularly for satisfying SOTIF requirements via waypoint accuracy metrics.
  C5 Fault coverage      : 4 | reason=Capable of characterizing highly nuanced anomaly types that rule-based systems cannot classify.
 
[R1 — exclusion]
  C1 Transparency        : 4 | reason=The decision to logically remove a sensor node is a binary, easily auditable action.
  C2 Computational cost  : 4 | reason=Logical exclusion from the fusion pipeline is a near-zero cost operation for the CPU.
  C3 Response time       : 4 | reason=Instantaneous modification of the sensor list ensures rapid system response to a fault.
  C4 Maturity            : 4 | reason=The most common and mature recovery mechanism used in redundant sensor architectures.
  C5 Fault coverage      : 3 | reason=Effectively masks faulty nodes but results in an immediate reduction of system redundancy and FOV.
 
[R2 — weight reconfiguration]
  C1 Transparency        : 3 | reason=The use of discounting factors is documentable, though the resulting fusion behavior is less predictable than exclusion.
  C2 Computational cost  : 3 | reason=Requires ongoing calculation of weakening factors based on real-time innovation residuals.
  C3 Response time       : 3 | reason=Depends on how quickly residuals are evaluated and applied to the fusion layer.
  C4 Maturity            : 3 | reason=Well-supported in research focusing on fuzzy logic and belief function-based sensor fusion.
  C5 Fault coverage      : 4 | reason=Provides superior fail-operational behavior by allowing degraded sensors to still contribute safely.
 
[R3 — bypass]
  C1 Transparency        : 4 | reason=The logic for switching between redundant KF branches is deterministic and easily traced.
  C2 Computational cost  : 3 | reason=Maintaining multiple Kalman Filter banks in parallel increases the system's memory footprint.
  C3 Response time       : 4 | reason=The KF Bank and Switch mechanism allows for near-instantaneous masking of hardware faults.
  C4 Maturity            : 4 | reason=Standard practice in high-integrity perception pipelines involving hardware-level analytical redundancy.
  C5 Fault coverage      : 3 | reason=Highly effective for hardware faults, provided that a redundant, healthy branch is physically available.

##### GPT-5.3

LLM: GPT-5.3
 
[D1 — model-based detection]
  C1 Transparency        : 4 | reason=Residual generation and state estimation are fully defined by explicit mathematical models (e.g., Kalman filters), enabling traceability.
  C2 Computational cost  : 3 | reason=Kalman filtering is efficient but still requires matrix operations and model propagation at each timestep.
  C3 Response time       : 4 | reason=Recursive estimation provides immediate residuals at each measurement update with minimal latency.
  C4 Maturity            : 4 | reason=Widely established and validated in AV perception and control systems literature.
  C5 Fault coverage      : 3 | reason=Effective for Gaussian noise and abrupt deviations but struggles with unmodeled dynamics and complex sensor faults.
 
[D2 — signal-based detection]
  C1 Transparency        : 4 | reason=Thresholds, filters, and statistical distances are explicitly defined and easily interpretable.
  C2 Computational cost  : 4 | reason=Operations like filtering and distance calculations are lightweight and suitable for embedded systems.
  C3 Response time       : 4 | reason=Direct processing of raw signals allows near-instant anomaly detection.
  C4 Maturity            : 4 | reason=Classical signal processing techniques are well-established and widely used.
  C5 Fault coverage      : 3 | reason=Handles noise and signal corruption well but is sensitive to environmental noise and threshold tuning.
 
[D3 — data-driven detection]
  C1 Transparency        : 1 | reason=Black-box neural network decisions lack interpretability and traceability for certification.
  C2 Computational cost  : 1 | reason=Requires significant computational resources and often specialized hardware (GPU/FPGA).
  C3 Response time       : 2 | reason=Inference latency is higher than classical methods, especially under embedded constraints.
  C4 Maturity            : 3 | reason=Demonstrated high performance but still faces certification and deployment barriers.
  C5 Fault coverage      : 3 | reason=Can detect complex anomalies but depends heavily on training data and struggles with unseen faults.
 
[D4 — active diagnosis]
  C1 Transparency        : 3 | reason=Test signal injection is conceptually clear but system response interpretation may be less explicit.
  C2 Computational cost  : 2 | reason=Additional processing and control overhead required for signal injection and analysis.
  C3 Response time       : 2 | reason=Detection requires time to inject and observe system response, introducing delay.
  C4 Maturity            : 2 | reason=Still evolving with limited adoption in real AV systems.
  C5 Fault coverage      : 3 | reason=Can distinguish certain faults effectively but depends on hardware capabilities and test design.
 
[I1 — output comparison]
  C1 Transparency        : 4 | reason=Comparison metrics like Mahalanobis distance are explicit and interpretable.
  C2 Computational cost  : 3 | reason=Requires cross-sensor alignment and matrix operations but remains manageable.
  C3 Response time       : 3 | reason=Dependent on synchronization and association steps, slightly increasing latency.
  C4 Maturity            : 4 | reason=Commonly used in multi-sensor fusion systems.
  C5 Fault coverage      : 3 | reason=Effective for detecting discrepancies but limited by reference sensor reliability.
 
[I2 — voting]
  C1 Transparency        : 4 | reason=Majority rules and consensus logic are simple and easily auditable.
  C2 Computational cost  : 3 | reason=Moderate overhead from multiple sensor evaluations and consensus computation.
  C3 Response time       : 3 | reason=Requires aggregation of multiple inputs before decision.
  C4 Maturity            : 4 | reason=Well-established in fault-tolerant systems.
  C5 Fault coverage      : 3 | reason=Handles isolated faults well but fails under common-cause failures.
 
[I3 — dependency modeling]
  C1 Transparency        : 3 | reason=Graph-based models are interpretable but become complex as system size grows.
  C2 Computational cost  : 2 | reason=Probabilistic inference over networks is computationally intensive.
  C3 Response time       : 2 | reason=Inference over large models introduces latency not ideal for real-time systems.
  C4 Maturity            : 3 | reason=Used in research and some applications but not universally deployed.
  C5 Fault coverage      : 4 | reason=Captures fault propagation and complex dependencies effectively.
 
[I4 — adaptive filters]
  C1 Transparency        : 3 | reason=Filter structure is known but adaptive behavior reduces full interpretability.
  C2 Computational cost  : 3 | reason=Moderate cost due to continuous parameter adaptation.
  C3 Response time       : 3 | reason=Operates online but requires adaptation time for accurate outputs.
  C4 Maturity            : 3 | reason=Established but less standardized than classical methods.
  C5 Fault coverage      : 3 | reason=Good for deviations from nominal behavior but sensitive to initialization and tuning.
 
[ID1 — rule-based identification]
  C1 Transparency        : 4 | reason=Explicit logical rules provide full interpretability and traceability.
  C2 Computational cost  : 4 | reason=Simple logical evaluations require minimal resources.
  C3 Response time       : 4 | reason=Immediate evaluation once inputs are available.
  C4 Maturity            : 4 | reason=Widely used in safety-critical systems.
  C5 Fault coverage      : 2 | reason=Limited to predefined fault cases and struggles with unseen anomalies.
 
[ID2 — statistical identification]
  C1 Transparency        : 3 | reason=Statistical metrics are interpretable but less intuitive than rule-based logic.
  C2 Computational cost  : 2 | reason=Dimensionality reduction and statistical computations are relatively heavy.
  C3 Response time       : 3 | reason=Moderate latency due to batch or window-based processing.
  C4 Maturity            : 4 | reason=Well-established in signal processing and fault diagnosis.
  C5 Fault coverage      : 3 | reason=Effective for distinguishing statistical deviations but limited by assumptions.
 
[ID3 — ML identification]
  C1 Transparency        : 1 | reason=Black-box classifiers lack explainability.
  C2 Computational cost  : 1 | reason=Requires significant computational resources and training infrastructure.
  C3 Response time       : 2 | reason=Inference latency higher than rule-based/statistical methods.
  C4 Maturity            : 3 | reason=Growing adoption but still limited by certification challenges.
  C5 Fault coverage      : 3 | reason=Good classification performance but limited to trained fault classes.
 
[R1 — exclusion]
  C1 Transparency        : 4 | reason=Decision to remove a sensor is explicit and easily traceable.
  C2 Computational cost  : 4 | reason=Simple logic with minimal runtime overhead.
  C3 Response time       : 4 | reason=Immediate action once fault is identified.
  C4 Maturity            : 4 | reason=Widely used in fault-tolerant systems.
  C5 Fault coverage      : 3 | reason=Effective for isolating faulty sensors but reduces system redundancy.
 
[R2 — weight reconfiguration]
  C1 Transparency        : 3 | reason=Weight adjustment is interpretable but depends on dynamic metrics.
  C2 Computational cost  : 3 | reason=Requires continuous evaluation of sensor reliability and weight updates.
  C3 Response time       : 3 | reason=Adjustments occur online but may lag behind rapid faults.
  C4 Maturity            : 3 | reason=Common in adaptive fusion but less standardized than exclusion.
  C5 Fault coverage      : 4 | reason=Maintains system operation across a wide range of fault conditions.
 
[R3 — bypass]
  C1 Transparency        : 4 | reason=Explicit rerouting of data flow is clearly defined and auditable.
  C2 Computational cost  : 3 | reason=Switching logic is simple but maintaining redundant paths adds overhead.
  C3 Response time       : 4 | reason=Fast switching mechanisms enable near-instant recovery.
  C4 Maturity            : 3 | reason=Used in practice but dependent on system architecture design.
  C5 Fault coverage      : 4 | reason=Highly effective when redundant paths are available for abrupt faults.

##### DeepSeek-R1

LLM: DeepSeek-R1
 
[D1 — model-based detection]
  C1 Transparency        : 4 | reason=Kalman filter equations and residual thresholds are fully deterministic and mathematically documented, enabling full auditability.
  C2 Computational cost  : 3 | reason=Kalman filter prediction/update steps involve moderate matrix operations but are tractable on embedded hardware.
  C3 Response time       : 4 | reason=Residual generation via single-step state estimation and innovation calculation yields near-instantaneous output.
  C4 Maturity            : 4 | reason=Model-based detection via Kalman filtering is a cornerstone technique extensively validated in AV state estimation and fault detection.
  C5 Fault coverage      : 4 | reason=Residuals directly reflect deviations from expected dynamics, effectively capturing data loss, Gaussian noise, dead-zone, and intensity corruption across heterogeneous sensors.
 
[D2 — signal-based detection]
  C1 Transparency        : 4 | reason=Explicit threshold-based statistical tests and transform parameters are fully documented and auditable.
  C2 Computational cost  : 3 | reason=Median filtering and simple statistical computations are lightweight; wavelet transforms add moderate overhead but remain feasible.
  C3 Response time       : 4 | reason=Signal processing and threshold comparison execute in near-real time with minimal latency.
  C4 Maturity            : 3 | reason=Signal-based anomaly detection is mature in industrial monitoring; its application in AV perception is validated but less dominant than model-based methods.
  C5 Fault coverage      : 4 | reason=Direct analysis of raw sensor streams readily identifies abrupt noise, dead-zone flat-lining, and intensity corruption across modalities.
 
[D3 — data-driven detection]
  C1 Transparency        : 1 | reason=Deep neural networks are black-box models with no formal guarantee or traceable decision path, violating ISO 26262 audit requirements.
  C2 Computational cost  : 1 | reason=CNNs, GANs, and autoencoders demand GPU/FPGA acceleration and far exceed typical embedded edge memory and processing budgets.
  C3 Response time       : 2 | reason=Deep inference introduces significant latency; even optimized accelerators cannot match the speed of analytical filters.
  C4 Maturity            : 2 | reason=High research accuracy contrasts with unresolved certification and verification gaps, leaving it immature for safety-critical AV deployment.
  C5 Fault coverage      : 3 | reason=Can learn diverse abrupt fault patterns when adequately trained, but OOD overconfidence and class imbalance limit coverage of unseen anomalies.
 
[D4 — active diagnosis]
  C1 Transparency        : 3 | reason=Deterministic test signal injection and response analysis provide a documented logic chain, though stochastic variants reduce full transparency.
  C2 Computational cost  : 2 | reason=Active probing temporarily increases sensor processing load and may reduce effective sampling rate, adding moderate overhead.
  C3 Response time       : 2 | reason=The injection-response-verification cycle introduces an intentional delay before fault confirmation, slowing detection.
  C4 Maturity            : 2 | reason=Active diagnosis is experimental and fragmented in AV perception; cited references show conflicting maturity and recent shift toward runtime monitors.
  C5 Fault coverage      : 2 | reason=Coverage depends on specially designed test signals; not all abrupt fault types can be probed without tailored and potentially impractical injection sequences.
 
[I1 — output comparison]
  C1 Transparency        : 4 | reason=Innovation-check and Mahalanobis distance with fixed thresholds provide a fully deterministic and auditable isolation logic.
  C2 Computational cost  : 3 | reason=Computing innovation vectors and Mahalanobis distances adds moderate arithmetic load, manageable on embedded processors.
  C3 Response time       : 4 | reason=Isolation via residual comparison and bipartite matching completes in a single processing step with negligible delay.
  C4 Maturity            : 4 | reason=Output comparison is a widely studied and applied technique for sensor fault isolation in AV fusion architectures.
  C5 Fault coverage      : 3 | reason=Effectively isolates abrupt sensor discrepancies when a reliable reference exists, but vulnerable to common-mode failures that affect both branches.
 
[I2 — voting]
  C1 Transparency        : 4 | reason=Majority consensus and Byzantine protocols use explicit, deterministic rules that are fully traceable.
  C2 Computational cost  : 4 | reason=Aggregating votes or averaging probabilities is computationally trivial, ideal for embedded real-time systems.
  C3 Response time       : 4 | reason=Consensus decision can be reached in a single cycle, providing immediate fault isolation.
  C4 Maturity            : 3 | reason=Voting is a classic fault-tolerance mechanism, but its use in heterogeneous AV sensor suites is tempered by documented common-cause failure vulnerabilities.
  C5 Fault coverage      : 3 | reason=Provides robust isolation for arbitrary faults under the n>=3f+1 assumption, but abrupt environmental corruption affecting multiple modalities may defeat the majority.
 
[I3 — dependency modeling]
  C1 Transparency        : 3 | reason=Bayesian networks and Petri nets are based on explicit causal graphs, though probabilistic inference paths can be complex to fully audit.
  C2 Computational cost  : 2 | reason=Probabilistic inference over large dependency graphs is computationally intensive, challenging hard real-time embedded constraints.
  C3 Response time       : 2 | reason=Graph-based fault propagation analysis requires iterative or probabilistic computation, adding latency to isolation.
  C4 Maturity            : 2 | reason=Dependency modeling with CPTPN and Bayesian nets is explored in AV research but lacks widespread production validation.
  C5 Fault coverage      : 3 | reason=Can theoretically trace any modeled fault origin, but coverage is contingent on exhaustive and accurate pre-modeling of all failure paths.
 
[I4 — adaptive filters]
  C1 Transparency        : 2 | reason=Adaptation logic and learned nominal profiles introduce dynamic, less interpretable states compared to static thresholds.
  C2 Computational cost  : 2 | reason=Continuous online learning and filter parameter re-estimation impose significant ongoing computational demand.
  C3 Response time       : 2 | reason=Requires a warm-up period to establish baseline and data accumulation for reliable deviation detection, delaying initial response.
  C4 Maturity            : 3 | reason=Adaptive filtering techniques are well-documented in research literature but have limited penetration into safety-certified AV perception pipelines.
  C5 Fault coverage      : 3 | reason=Can isolate sensors exhibiting abrupt deviation from learned nominal behavior, though warm-up may miss early faults.
 
[ID1 — rule-based identification]
  C1 Transparency        : 4 | reason=Explicit if-then logical conditions and fault taxonomies are completely auditable and certifiable.
  C2 Computational cost  : 4 | reason=Boolean rule evaluation is computationally negligible, perfectly suited to embedded real-time constraints.
  C3 Response time       : 4 | reason=Rule matching produces a categorical fault type in microseconds.
  C4 Maturity            : 4 | reason=Rule-based expert systems are a mature, widely deployed diagnostic approach in automotive systems.
  C5 Fault coverage      : 2 | reason=Coverage is limited to pre-enumerated fault signatures; novel or unanticipated abrupt fault patterns will not be identified.
 
[ID2 — statistical identification]
  C1 Transparency        : 3 | reason=PCA and Mahalanobis distance are mathematically defined, but interpretation of reduced dimensions requires domain expertise, slightly reducing direct auditability.
  C2 Computational cost  : 2 | reason=Eigenvalue decomposition or iterative ICA estimation poses non-trivial computational load for embedded processors.
  C3 Response time       : 3 | reason=Statistical analysis can operate on sliding windows with acceptable latency, though batch processing may introduce small delays.
  C4 Maturity            : 3 | reason=Statistical identification is established in fault diagnosis, with demonstrated but not dominant use in AV perception identification.
  C5 Fault coverage      : 3 | reason=Quantifiable deviation metrics effectively distinguish abrupt noise and dead-zone faults from nominal distributions, but may misclassify overlapping artifacts.
 
[ID3 — ML identification]
  C1 Transparency        : 1 | reason=Neural network classifiers are black-box models with no traceable decision rationale, fundamentally incompatible with ISO 26262 auditability.
  C2 Computational cost  : 1 | reason=Supervised classifiers require GPU/FPGA acceleration, violating embedded real-time processing and resource constraints.
  C3 Response time       : 2 | reason=Inference latency, even with acceleration, adds non-negligible delay unsuitable for hard real-time identification.
  C4 Maturity            : 2 | reason=ML fault identification remains research-grade, hindered by scarce annotated fault data and unresolved safety certification.
  C5 Fault coverage      : 2 | reason=Effective only for abrupt faults present in training; cannot identify novel or out-of-distribution fault types.
 
[R1 — exclusion]
  C1 Transparency        : 4 | reason=Removing a sensor from the fusion list is a simple, deterministic operation with fully auditable decision logging.
  C2 Computational cost  : 4 | reason=Exclusion costs virtually zero additional computation, only requiring pointer or list modification.
  C3 Response time       : 4 | reason=Reconfiguration is instantaneous upon reception of the isolation signal.
  C4 Maturity            : 4 | reason=Sensor exclusion is a standard, field-proven recovery mechanism in fault-tolerant AV sensor fusion.
  C5 Fault coverage      : 3 | reason=Restores safe operation if redundant sensors remain, but coverage degrades as exclusions reduce FOV or modality diversity.
 
[R2 — weight reconfiguration]
  C1 Transparency        : 3 | reason=Discounting factor formulas and weight update rules are explicitly specified, though the dynamic interplay in fusion is more complex to audit.
  C2 Computational cost  : 3 | reason=Computing innovation-based weights adds small additional filtering steps but remains within embedded capability.
  C3 Response time       : 4 | reason=Weight recalculation occurs within the same fusion cycle, adding negligible latency.
  C4 Maturity            : 3 | reason=Belief-function discounting and dynamic weighting are well-researched and partially adopted, though less universally standard than exclusion.
  C5 Fault coverage      : 3 | reason=Gradually reduces influence of noisy or partially faulty sensors, effectively handling moderate abrupt faults without full data loss, but risk of slow response to fast corruption.
 
[R3 — bypass]
  C1 Transparency        : 4 | reason=Switching to a redundant sensor branch is a deterministic path reconfiguration that is straightforward to trace.
  C2 Computational cost  : 4 | reason=Redirecting data flow is logically equivalent to a multiplexer operation with negligible overhead.
  C3 Response time       : 4 | reason=Bypass switching occurs immediately upon trigger, maintaining uninterrupted perception output.
  C4 Maturity            : 4 | reason=Redundant path switching (KF bank and switch) is a proven concept in fault-tolerant AV perception systems.
  C5 Fault coverage      : 2 | reason=Coverage is strictly limited to faults for which a dedicated redundant sensor branch exists; many sensor faults have no alternative path within perception.

#### Vetor A — Scores de qualidade das DAs

Score inicial por média arredondada de 4 LLMs (Claude Opus 4.6, Gemini 3.1 Pro, GPT-5.3, DeepSeek).  
Escala 1–5 por critério. C2 e C3 com escala invertida (5 = melhor).

In [106]:
VETOR_A_LLMS = {
    'Claude': {
        'D1':  [4, 3, 4, 4, 3],
        'D2':  [4, 3, 4, 3, 3],
        'D3':  [1, 1, 2, 2, 3],
        'D4':  [3, 3, 2, 2, 2],
        'I1':  [4, 3, 3, 4, 4],
        'I2':  [4, 2, 3, 4, 3],
        'I3':  [3, 2, 2, 3, 2],
        'I4':  [3, 3, 3, 3, 3],
        'ID1': [4, 4, 4, 4, 3],
        'ID2': [3, 2, 3, 3, 3],
        'ID3': [1, 1, 2, 2, 3],
        'R1':  [4, 4, 4, 4, 3],
        'R2':  [3, 3, 3, 3, 4],
        'R3':  [4, 4, 4, 3, 3],
    },
    'Gemini': {
        'D1':  [4, 3, 3, 4, 3],
        'D2':  [4, 4, 4, 4, 2],
        'D3':  [1, 1, 2, 3, 4],
        'D4':  [3, 3, 2, 2, 3],
        'I1':  [4, 3, 3, 4, 3],
        'I2':  [4, 4, 4, 4, 3],
        'I3':  [3, 2, 2, 3, 3],
        'I4':  [3, 2, 2, 3, 4],
        'ID1': [4, 4, 4, 4, 2],
        'ID2': [3, 2, 3, 3, 3],
        'ID3': [1, 1, 2, 3, 4],
        'R1':  [4, 4, 4, 4, 3],
        'R2':  [3, 3, 3, 3, 4],
        'R3':  [4, 3, 4, 4, 3],
    },
    'GPT': {
        'D1':  [4, 3, 4, 4, 3],
        'D2':  [4, 4, 4, 4, 3],
        'D3':  [1, 1, 2, 3, 3],
        'D4':  [3, 2, 2, 2, 3],
        'I1':  [4, 3, 3, 4, 3],
        'I2':  [4, 3, 3, 4, 3],
        'I3':  [3, 2, 2, 3, 4],
        'I4':  [3, 3, 3, 3, 3],
        'ID1': [4, 4, 4, 4, 2],
        'ID2': [3, 2, 3, 4, 3],
        'ID3': [1, 1, 2, 3, 3],
        'R1':  [4, 4, 4, 4, 3],
        'R2':  [3, 3, 3, 3, 4],
        'R3':  [4, 3, 4, 3, 4],
    },
    'DeepSeek': {
        'D1':  [4, 3, 4, 4, 4],
        'D2':  [4, 3, 4, 3, 4],
        'D3':  [1, 1, 2, 2, 3],
        'D4':  [3, 2, 2, 2, 2],
        'I1':  [4, 3, 4, 4, 3],
        'I2':  [4, 4, 4, 3, 3],
        'I3':  [3, 2, 2, 2, 3],
        'I4':  [2, 2, 2, 3, 3],
        'ID1': [4, 4, 4, 4, 2],
        'ID2': [3, 2, 3, 3, 3],
        'ID3': [1, 1, 2, 2, 2],
        'R1':  [4, 4, 4, 4, 3],
        'R2':  [3, 3, 4, 3, 3],
        'R3':  [4, 4, 4, 4, 2],
    }
}

llm_names = list(VETOR_A_LLMS.keys())
das = list(VETOR_A_LLMS['Claude'].keys())

# Calcular media e desvio padrao por celula
records = []
for da in das:
    scores = np.array([VETOR_A_LLMS[llm][da] for llm in llm_names])
    means = scores.mean(axis=0)
    stds  = scores.std(axis=0)
    final = np.round(means).astype(int)
    records.append({
        'DA': da,
        'C1_final': final[0], 'C1_mean': means[0], 'C1_std': stds[0],
        'C2_final': final[1], 'C2_mean': means[1], 'C2_std': stds[1],
        'C3_final': final[2], 'C3_mean': means[2], 'C3_std': stds[2],
        'C4_final': final[3], 'C4_mean': means[3], 'C4_std': stds[3],
        'C5_final': final[4], 'C5_mean': means[4], 'C5_std': stds[4],
    })

df_consolidado = pd.DataFrame(records).set_index('DA')

# Tabela de scores finais
print("Vetor A -- Score Final (media arredondada)")
print("-" * 50)
cols_final = ['C1_final','C2_final','C3_final','C4_final','C5_final']
df_final = df_consolidado[cols_final].copy()
df_final.columns = CRITERIA
df_final['Sigma'] = df_final.sum(axis=1)
print(df_final.to_string())

# Celulas com divergencia (std > 0.5)
print("\nCelulas com divergencia (std > 0.5):")
print(f"{'DA':<6} {'Criterio':<12} {'Claude':>7} {'Gemini':>7} {'GPT':>7} "
      f"{'DeepSeek':>9} {'Media':>7} {'Std':>6} {'Final':>6}")
print("-" * 70)
for da in das:
    scores = np.array([VETOR_A_LLMS[llm][da] for llm in llm_names])
    for i, c in enumerate(CRITERIA):
        std = scores[:, i].std()
        if std > 0.5:
            mean = scores[:, i].mean()
            final = round(mean)
            print(f"{da:<6} {c:<12} "
                  f"{scores[0,i]:>7} {scores[1,i]:>7} {scores[2,i]:>7} "
                  f"{scores[3,i]:>9} {mean:>7.2f} {std:>6.2f} {final:>6}")

Vetor A -- Score Final (media arredondada)
--------------------------------------------------
     C1  C2  C3  C4  C5  Sigma
DA                            
D1    4   3   4   4   3     18
D2    4   4   4   4   3     19
D3    1   1   2   2   3      9
D4    3   2   2   2   2     11
I1    4   3   3   4   3     17
I2    4   3   4   4   3     18
I3    3   2   2   3   3     13
I4    3   2   2   3   3     13
ID1   4   4   4   4   2     18
ID2   3   2   3   3   3     14
ID3   1   1   2   2   3      9
R1    4   4   4   4   3     19
R2    3   3   3   3   4     16
R3    4   4   4   4   3     19

Celulas com divergencia (std > 0.5):
DA     Criterio      Claude  Gemini     GPT  DeepSeek   Media    Std  Final
----------------------------------------------------------------------
D2     C5                 3       2       3         4    3.00   0.71      3
I2     C2                 2       4       3         4    3.25   0.83      3
I3     C5                 2       3       4         3    3.00   0.71     

#### Prompt de Revisão de Discordâncias

~~~
================================================================================
REVISION PROMPT — VECTOR A DIVERGENT CELLS
================================================================================

IDENTIFICATION:
Before providing any assessment, identify yourself on the first line:
  LLM: [your model name and version]
Then proceed with the full revision.

CONTEXT:
You previously evaluated 14 FDIIR design alternatives across 5 criteria
(scale 1-4). The following 5 cells showed score divergence (std > 0.5)
across the four evaluators. For each cell, the scores and justifications
from all four evaluators are provided below.

SYSTEM CONSTRAINTS (unchanged):
  - Pipeline: Detection -> Isolation -> Identification -> Recovery
  - Platform: ROS 2 / Autoware Universe, local embedded processing
  - Hard real-time constraints apply
  - Fault scope: abrupt faults only (data loss, Gaussian noise, dead zone,
    intensity corruption) — gradual/incipient faults are OUT OF SCOPE
  - System scope: all modules operate exclusively within the perception
    system. Planning-level decisions are OUT OF SCOPE

SCALE (unchanged):
  1 = very poor
  2 = poor
  3 = good
  4 = excellent

TASK:
For each divergent cell below, review all four positions and provide:
  - Your final score: one value from {1, 2, 3, 4}
  - Decision: MAINTAIN (same as your original) or REVISE
  - One-sentence rationale for your final position

DIVERGENT CELLS:

--------------------------------------------------------------------------------
[D2 - C5 Fault coverage]
Criterion: adequacy for abrupt faults (data loss, Gaussian noise, dead zone,
intensity corruption) in a heterogeneous sensor suite (camera + LiDAR + Radar)

  Claude:   3 | fixed thresholds can confuse environmental noise with genuine
              faults, but strong fit for intensity corruption, dead zones,
              and impulse noise
  Gemini:   2 | susceptible to environmental noise overlaps and fails to
              account for dynamic system inputs
  GPT:      3 | handles noise and signal corruption well but sensitive to
              environmental noise and threshold tuning
  DeepSeek: 4 | direct analysis of raw sensor streams readily identifies
              abrupt noise, dead-zone flat-lining, and intensity corruption
              across modalities

--------------------------------------------------------------------------------
[I2 - C2 Computational cost (inverted)]
Criterion: CPU/memory resources at runtime — 4 = lightweight, 1 = heavy

  Claude:   2 | N-version execution multiplies compute and memory footprint,
              straining non-intensive embedded budgets
  Gemini:   4 | majority decision logic is computationally trivial, though
              it assumes the presence of physical hardware replicas
  GPT:      3 | moderate overhead from multiple sensor evaluations and
              consensus computation
  DeepSeek: 4 | aggregating votes or averaging probabilities is
              computationally trivial, ideal for embedded real-time systems

--------------------------------------------------------------------------------
[I3 - C5 Fault coverage]
Criterion: adequacy for abrupt faults (data loss, Gaussian noise, dead zone,
intensity corruption) in a heterogeneous sensor suite (camera + LiDAR + Radar)

  Claude:   2 | documented difficulty separating true anomalies from
              environmental contamination narrows abrupt-fault coverage
  Gemini:   3 | superior for mapping fault propagation, though it struggles
              with environmental contamination signatures
  GPT:      4 | captures fault propagation and complex dependencies effectively
  DeepSeek: 3 | can theoretically trace any modeled fault origin, but coverage
              is contingent on exhaustive and accurate pre-modeling

--------------------------------------------------------------------------------
[ID3 - C5 Fault coverage]
Criterion: adequacy for abrupt faults (data loss, Gaussian noise, dead zone,
intensity corruption) in a heterogeneous sensor suite (camera + LiDAR + Radar)

  Claude:   3 | explicit Fault Class and incremental learning cover the
              in-scope abrupt set when labels exist, but unseen variants
              are missed
  Gemini:   4 | capable of characterizing highly nuanced anomaly types that
              rule-based systems cannot classify
  GPT:      3 | good classification performance but limited to trained fault
              classes
  DeepSeek: 2 | effective only for abrupt faults present in training; cannot
              identify novel or out-of-distribution fault types

--------------------------------------------------------------------------------
[R3 - C5 Fault coverage]
Criterion: adequacy for abrupt faults (data loss, Gaussian noise, dead zone,
intensity corruption) in a heterogeneous sensor suite (camera + LiDAR + Radar)

  Claude:   3 | covers abrupt faults cleanly when a redundant path exists,
              but coverage collapses where redundant hardware is absent
  Gemini:   3 | highly effective for hardware faults, provided that a
              redundant, healthy branch is physically available
  GPT:      4 | highly effective when redundant paths are available for
              abrupt faults
  DeepSeek: 2 | coverage is strictly limited to faults for which a dedicated
              redundant sensor branch exists; many sensor faults have no
              alternative path within perception

RESPONSE FORMAT:
  [D2 - C5]  Final: X | Decision: MAINTAIN/REVISE | reason=...
  [I2 - C2]  Final: X | Decision: MAINTAIN/REVISE | reason=...
  [I3 - C5]  Final: X | Decision: MAINTAIN/REVISE | reason=...
  [ID3 - C5] Final: X | Decision: MAINTAIN/REVISE | reason=...
  [R3 - C5]  Final: X | Decision: MAINTAIN/REVISE | reason=...
================================================================================
~~~

#### Respostas — Prompt de revisão

LLM: DeepSeek-R1
  [D2 - C5]  Final: 3 | Decision: REVISE | reason=Signal-based methods readily detect dead-zone, data-loss, and impulse faults but fixed thresholds in noisy environments may miss some abrupt deviations, preventing maximum coverage.
  [I2 - C2]  Final: 4 | Decision: MAINTAIN | reason=The voting aggregation step itself is computationally trivial; the runtime cost of generating redundant inputs belongs to sensor processing, not the isolation logic.
  [I3 - C5]  Final: 3 | Decision: MAINTAIN | reason=Dependency models can theoretically capture all specified abrupt faults if correctly modeled, but exhaustive pre-modeling is a practical barrier that limits full assurance.
  [ID3 - C5] Final: 3 | Decision: REVISE | reason=ML classifiers can be trained on labeled examples of data loss, noise, dead-zone, and corruption, providing good coverage for known patterns, though they remain blind to untrained variants.
  [R3 - C5]  Final: 2 | Decision: MAINTAIN | reason=Bypass coverage is inherently conditional on the availability of a dedicated redundant perception path, which is not guaranteed for all sensor types and thus remains poor in the general case.

LLM: GPT-5.3
[D2 - C5] Final: 3 | Decision: MAINTAIN | reason=Signal-based methods reliably detect abrupt artifacts like noise spikes, dead zones, and intensity corruption, but their sensitivity to environmental noise and static thresholds prevents excellent coverage.
[I2 - C2] Final: 3 | Decision: MAINTAIN | reason=While the voting logic itself is lightweight, the need to process multiple redundant sensor streams introduces non-negligible runtime overhead in embedded systems.
[I3 - C5] Final: 3 | Decision: REVISE | reason=Fault coverage is fundamentally limited by the completeness and accuracy of the modeled dependencies, making it good but not exhaustive for abrupt faults.
[ID3 - C5] Final: 3 | Decision: MAINTAIN | reason=ML classifiers can accurately identify abrupt faults within the trained set, but inability to generalize to unseen fault patterns constrains overall coverage.
[R3 - C5] Final: 3 | Decision: REVISE | reason=Bypass is highly effective for abrupt faults only when redundant paths exist, but its dependence on architectural redundancy limits universal fault coverage.

LLM: Gemini 2.0 Flash
[D2 - C5] Final: 3 | Decision: REVISE | reason=While environmental noise is a factor, signal-based methods like MD gating and Median filtering are inherently well-suited for identifying the specific abrupt fault set (dead zones, data loss, and outliers) at the signal level.
[I2 - C2] Final: 4 | Decision: MAINTAIN | reason=While N-version systems require more total hardware, the isolation logic itself (majority voting or probability averaging) is computationally negligible and ideal for hard real-time embedded constraints.
[I3 - C5] Final: 3 | Decision: MAINTAIN | reason=Bayesian and Petri nets provide excellent coverage for modeled abrupt fault propagation, though the "modeling bottleneck" and difficulty with environmental noise prevent a perfect score.
[ID3 - C5] Final: 3 | Decision: REVISE | reason=ML identification is powerful for characterizing nuanced abrupt faults like intensity corruption, but coverage is strictly limited to the labels present in the training data, as noted in the source context.
[R3 - C5] Final: 3 | Decision: MAINTAIN | reason=The KF Bank and Switch mechanism is highly effective for masking the abrupt loss of hardware, though its overall coverage is physically constrained by the availability of redundant sensor branches.

LLM: Claude Opus 4.7 (Anthropic)
[D2 - C5] Final: 3 | Decision: MAINTAIN | reason=Raw-signal methods (median filter, DWT, Mahalanobis gating) directly target the in-scope abrupt faults across camera/LiDAR/radar, but the documented brittleness to environmental noise overlap (Koopman & Wagner; Goelles) prevents a top score of 4.
[I2 - C2] Final: 3 | Decision: REVISE | reason=The voting aggregation itself (majority decision, soft-voting averaging) is computationally trivial, but N-version / self-checking programming and synchronization overhead add non-negligible runtime load on embedded platforms, placing it between the extremes.
[I3 - C5] Final: 3 | Decision: REVISE | reason=For the bounded abrupt fault set within a pre-modeled topology, Bayesian networks / CPTPN / SDG do trace propagation effectively; coverage is reasonable and the primary caveat is the exhaustive-modeling prerequisite rather than the abrupt-fault nature itself.
[ID3 - C5] Final: 3 | Decision: MAINTAIN | reason=Supervised classifiers handle the well-defined abrupt fault classes (data loss, Gaussian noise, dead zone, intensity corruption) competently with sufficient labels and incremental learning, but the explicit "cannot identify faults not encountered during training" limitation prevents a 4.
[R3 - C5] Final: 3 | Decision: MAINTAIN | reason=Within the heterogeneous camera+LiDAR+radar suite there is usually a viable redundant branch for abrupt faults, giving solid coverage; however, DeepSeek's concern is valid where no alternative path exists, which keeps coverage bounded rather than excellent.

#### Vetor A — Consolidação

In [107]:
VETOR_A_REVISADO = {
    'Claude': {
        'D1':  [4, 3, 4, 4, 3],
        'D2':  [4, 3, 4, 3, 3],
        'D3':  [1, 1, 2, 2, 3],
        'D4':  [3, 3, 2, 2, 2],
        'I1':  [4, 3, 3, 4, 4],
        'I2':  [4, 3, 3, 4, 3],  # C2: 2->3
        'I3':  [3, 2, 2, 3, 3],  # C5: 2->3
        'I4':  [3, 3, 3, 3, 3],
        'ID1': [4, 4, 4, 4, 3],
        'ID2': [3, 2, 3, 3, 3],
        'ID3': [1, 1, 2, 2, 3],
        'R1':  [4, 4, 4, 4, 3],
        'R2':  [3, 3, 3, 3, 4],
        'R3':  [4, 4, 4, 3, 3],
    },
    'Gemini': {
        'D1':  [4, 3, 3, 4, 3],
        'D2':  [4, 4, 4, 4, 3],  # C5: 2->3
        'D3':  [1, 1, 2, 3, 4],
        'D4':  [3, 3, 2, 2, 3],
        'I1':  [4, 3, 3, 4, 3],
        'I2':  [4, 4, 4, 4, 3],
        'I3':  [3, 2, 2, 3, 3],
        'I4':  [3, 2, 2, 3, 4],
        'ID1': [4, 4, 4, 4, 2],
        'ID2': [3, 2, 3, 3, 3],
        'ID3': [1, 1, 2, 3, 3],  # C5: 4->3
        'R1':  [4, 4, 4, 4, 3],
        'R2':  [3, 3, 3, 3, 4],
        'R3':  [4, 3, 4, 4, 3],
    },
    'GPT': {
        'D1':  [4, 3, 4, 4, 3],
        'D2':  [4, 4, 4, 4, 3],
        'D3':  [1, 1, 2, 3, 3],
        'D4':  [3, 2, 2, 2, 3],
        'I1':  [4, 3, 3, 4, 3],
        'I2':  [4, 3, 3, 4, 3],
        'I3':  [3, 2, 2, 3, 3],  # C5: 4->3
        'I4':  [3, 3, 3, 3, 3],
        'ID1': [4, 4, 4, 4, 2],
        'ID2': [3, 2, 3, 4, 3],
        'ID3': [1, 1, 2, 3, 3],
        'R1':  [4, 4, 4, 4, 3],
        'R2':  [3, 3, 3, 3, 4],
        'R3':  [4, 3, 4, 3, 3],  # C5: 4->3
    },
    'DeepSeek': {
        'D1':  [4, 3, 4, 4, 4],
        'D2':  [4, 3, 4, 3, 3],  # C5: 4->3
        'D3':  [1, 1, 2, 2, 3],
        'D4':  [3, 2, 2, 2, 2],
        'I1':  [4, 3, 4, 4, 3],
        'I2':  [4, 4, 4, 3, 3],
        'I3':  [3, 2, 2, 2, 3],
        'I4':  [2, 2, 2, 3, 3],
        'ID1': [4, 4, 4, 4, 2],
        'ID2': [3, 2, 3, 3, 3],
        'ID3': [1, 1, 2, 2, 3],  # C5: 2->3
        'R1':  [4, 4, 4, 4, 3],
        'R2':  [3, 3, 4, 3, 3],
        'R3':  [4, 4, 4, 4, 2],
    }
}

llm_names = list(VETOR_A_REVISADO.keys())
das = list(VETOR_A_REVISADO['Claude'].keys())

# Recalcular
records = []
for da in das:
    scores = np.array([VETOR_A_REVISADO[llm][da] for llm in llm_names])
    means = scores.mean(axis=0)
    stds  = scores.std(axis=0)
    final = np.round(means).astype(int)
    records.append({
        'DA': da,
        'C1_final': final[0], 'C1_std': stds[0],
        'C2_final': final[1], 'C2_std': stds[1],
        'C3_final': final[2], 'C3_std': stds[2],
        'C4_final': final[3], 'C4_std': stds[3],
        'C5_final': final[4], 'C5_std': stds[4],
    })

df_final_rev = pd.DataFrame(records).set_index('DA')
cols_final = ['C1_final','C2_final','C3_final','C4_final','C5_final']
vetor_a = df_final_rev[cols_final].copy()
vetor_a.columns = CRITERIA
vetor_a['Sigma'] = vetor_a.sum(axis=1)

print("Vetor A Final -- pos-revisao")
print("-" * 50)
print(vetor_a.to_string())

# Verificar se restam celulas com divergencia
print("\nVerificacao final -- celulas com std > 0.5:")
divergentes = []
for da in das:
    scores = np.array([VETOR_A_REVISADO[llm][da] for llm in llm_names])
    for i, c in enumerate(CRITERIA):
        std = scores[:, i].std()
        if std > 0.5:
            divergentes.append((da, c, std))

if divergentes:
    for da, c, std in divergentes:
        print(f"  {da} x {c}: std={std:.2f}")
else:
    print("  Nenhuma celula com divergencia remanescente.")

Vetor A Final -- pos-revisao
--------------------------------------------------
     C1  C2  C3  C4  C5  Sigma
DA                            
D1    4   3   4   4   3     18
D2    4   4   4   4   3     19
D3    1   1   2   2   3      9
D4    3   2   2   2   2     11
I1    4   3   3   4   3     17
I2    4   4   4   4   3     19
I3    3   2   2   3   3     13
I4    3   2   2   3   3     13
ID1   4   4   4   4   2     18
ID2   3   2   3   3   3     14
ID3   1   1   2   2   3      9
R1    4   4   4   4   3     19
R2    3   3   3   3   4     16
R3    4   4   4   4   3     19

Verificacao final -- celulas com std > 0.5:
  Nenhuma celula com divergencia remanescente.


#### Prompt B - Cross-Consistency Matrix

~~~
================================================================================
EVALUATION PROMPT B — VECTOR B (CROSS-CONSISTENCY MATRIX)
================================================================================

IDENTIFICATION:
Before providing any assessment, identify yourself on the first line:
  LLM: [your model name and version]
Then proceed with the full evaluation.

CONTEXT:
Based on the technical context provided, evaluate the pairwise
compatibility between design alternatives (DAs) from different modules.
Each cell represents an architectural question: when these two methods
are combined in the same FDIIR pipeline, how well do they coexist?

SYSTEM CONSTRAINTS (fixed for all assessments):
  - Pipeline: Detection -> Isolation -> Identification -> Recovery
  - Platform: ROS 2 / Autoware Universe, local embedded processing
  - Hard real-time constraints apply
  - Fault scope: abrupt faults only (data loss, Gaussian noise, dead zone,
    intensity corruption) — gradual/incipient faults are OUT OF SCOPE
  - System scope: all modules operate exclusively within the perception
    system. Planning-level decisions are OUT OF SCOPE

SCALE:
  0 = incompatible — logical or empirical contradiction; the pair
      cannot coexist in the same pipeline
  1 = compatible with severe restrictions — significant architectural
      conflict requiring major design intervention to resolve
  2 = compatible with moderate restrictions — identifiable conflict
      requiring specific design decision to resolve
  3 = fully compatible — no architectural restriction; the pair
      coexists without constraints

INSTRUCTIONS:
  1. Evaluate each pair independently.
  2. Assign exactly one value from {0, 1, 2, 3} per cell.
  3. Provide a one-sentence justification per cell describing the
     specific architectural reason for the score.
  4. Evaluate from first principles based on the context provided.
     Do not reference previous evaluations or existing scores.
  5. Compatibility is a property of the pair — not of individual DAs.
     Do not penalize a pair because one DA scores poorly on a criterion.
     Penalize only when combining the two creates a conflict that
     neither DA has alone.

PAIRS TO EVALUATE (6 blocks, 73 cells total):

BLOCK B1 — D x R (12 cells):
  D1 x R1 | D1 x R2 | D1 x R3
  D2 x R1 | D2 x R2 | D2 x R3
  D3 x R1 | D3 x R2 | D3 x R3
  D4 x R1 | D4 x R2 | D4 x R3

BLOCK B2 — D x ID (12 cells):
  D1 x ID1 | D1 x ID2 | D1 x ID3
  D2 x ID1 | D2 x ID2 | D2 x ID3
  D3 x ID1 | D3 x ID2 | D3 x ID3
  D4 x ID1 | D4 x ID2 | D4 x ID3

BLOCK B3 — D x I (16 cells):
  D1 x I1 | D1 x I2 | D1 x I3 | D1 x I4
  D2 x I1 | D2 x I2 | D2 x I3 | D2 x I4
  D3 x I1 | D3 x I2 | D3 x I3 | D3 x I4
  D4 x I1 | D4 x I2 | D4 x I3 | D4 x I4

BLOCK B4 — I x R (12 cells):
  I1 x R1 | I1 x R2 | I1 x R3
  I2 x R1 | I2 x R2 | I2 x R3
  I3 x R1 | I3 x R2 | I3 x R3
  I4 x R1 | I4 x R2 | I4 x R3

BLOCK B5 — I x ID (12 cells):
  I1 x ID1 | I1 x ID2 | I1 x ID3
  I2 x ID1 | I2 x ID2 | I2 x ID3
  I3 x ID1 | I3 x ID2 | I3 x ID3
  I4 x ID1 | I4 x ID2 | I4 x ID3

BLOCK B6 — ID x R (9 cells):
  ID1 x R1 | ID1 x R2 | ID1 x R3
  ID2 x R1 | ID2 x R2 | ID2 x R3
  ID3 x R1 | ID3 x R2 | ID3 x R3

RESPONSE FORMAT (one line per cell):
  [B1] D1 x R1 : X | reason=...
  [B1] D1 x R2 : X | reason=...
  [B1] D1 x R3 : X | reason=...
  ... (continue for all 73 cells in block order)
================================================================================
~~~

#### Respostas — Prompt B

##### Claude Opus 4.7

~~~
LLM: Claude Opus 4.7 (Anthropic)

**BLOCK B1 — D × R**

[B1] D1 x R1 : 3 | reason=D1 residuals directly flag the faulty sensor, and the KF Bank/Switch primitive uses that signal to exclude it from fusion with no architectural friction.
[B1] D1 x R2 : 3 | reason=Bader et al. documents the explicit dependency — innovation residuals from D1 feed the weakening factors p_trust that R2 applies, making them natural companions.
[B1] D1 x R3 : 3 | reason=Grubmüller documents D1 residuals as the primary trigger for R3 KF bank switching; the pair is by design.
[B1] D2 x R1 : 3 | reason=Raw-signal anomaly flags from D2 map cleanly onto a fusion-manager exclusion decision without conflict.
[B1] D2 x R2 : 2 | reason=R2 classically expects innovation residuals (as in Bader) rather than threshold-based signal flags, so translating D2 health metrics into graded weights requires a specific mapping layer.
[B1] D2 x R3 : 3 | reason=Signal-level fault flags localize the faulty branch well enough to drive KF Bank switching without architectural tension.
[B1] D3 x R1 : 3 | reason=Anomaly scores and OOD flags from D3 feed cleanly into exclusion logic; the black-box certification concern is a property of D3 alone, not of the pairing.
[B1] D3 x R2 : 3 | reason=GAN/DSAE confidence scores map naturally onto continuous weighting factors, enabling graded discounting rather than binary decisions.
[B1] D3 x R3 : 3 | reason=Nitsch et al. explicitly documents low-confidence GAN outputs flagging samples for recovery intervention, aligning with R3 semantics.
[B1] D4 x R1 : 2 | reason=D4's periodic probing introduces test-interval latency between fault occurrence and exclusion trigger, requiring a timing reconciliation layer to meet hard real-time constraints.
[B1] D4 x R2 : 3 | reason=Goelles explicitly documents D4 verifying whether R2 weight reconfiguration restored functionality — a direct documented synergy.
[B1] D4 x R3 : 3 | reason=D4 probes can both trigger and post-verify bypass routing without any inherent architectural contradiction.

**BLOCK B2 — D × ID**

[B2] D1 x ID1 : 3 | reason=Bader's four formal conditions apply rule thresholds directly to BetP values derived from D1, so the pipeline is natively designed around this pairing.
[B2] D1 x ID2 : 3 | reason=Innovation sequences and Conflict factor K from D1 are exactly the statistical primitives ID2 consumes (Mahalanobis, hypothesis testing).
[B2] D1 x ID3 : 3 | reason=Gao et al. documents model-based residuals as high-signal features for downstream classifiers — a direct synergy.
[B2] D2 x ID1 : 3 | reason=Signal-based anomaly flags and threshold crossings map directly onto rule-based logical conditions.
[B2] D2 x ID2 : 3 | reason=DWT/PSD/Mahalanobis outputs from D2 are inherently statistical and flow natively into ID2's hypothesis testing and PCA pipelines.
[B2] D2 x ID3 : 2 | reason=Raw-signal features can feed ML, but ID3 typically expects pre-processed feature vectors and labeled fault data, requiring an explicit feature-engineering layer over D2's raw outputs.
[B2] D3 x ID1 : 2 | reason=D3's continuous anomaly scores must be discretized to drive ID1's boolean gates, and the black-box output lacks the taxonomy alignment ID1 assumes.
[B2] D3 x ID2 : 3 | reason=D3 bottleneck features combine naturally with PCA/Mahalanobis post-hoc statistics (Nitsch et al.), a documented configuration.
[B2] D3 x ID3 : 3 | reason=Gao et al. documents the D3+ID3 pairing with PCA feature extraction to reduce computational overhead — a designed combination.
[B2] D4 x ID1 : 3 | reason=D4 produces discrete fault-mode outcomes (verified vs. discarded returns) that map cleanly onto ID1's predefined taxonomy rules.
[B2] D4 x ID2 : 3 | reason=Probability ratio tests intrinsic to D4 feed directly into ID2's hypothesis-testing framework.
[B2] D4 x ID3 : 2 | reason=Active probe responses are sparse and event-driven rather than continuous feature streams, requiring an adaptation layer before ID3 classification is viable.

**BLOCK B3 — D × I**

[B3] D1 x I1 : 3 | reason=Doc 15 documents D1 residuals as the primary baseline for I1 comparison logic — the canonical model-based isolation path.
[B3] D1 x I2 : 3 | reason=Parallel D1 instances across sensors support N-version voting directly, with residuals providing independent judgment per voter.
[B3] D1 x I3 : 3 | reason=D1 residuals naturally populate Bayesian network evidence nodes with quantitative probabilities aligned with I3's structure.
[B3] D1 x I4 : 3 | reason=Doc 13 documents D1 residuals as baseline for I4 adaptive filtering — an explicit synergy.
[B3] D2 x I1 : 3 | reason=Doc 15 documents the dependency — D2 signal deviations feed directly into I1 cross-sensor validation logic.
[B3] D2 x I2 : 3 | reason=Signal-based per-sensor detection runs independently per voter, aligning naturally with heterogeneous majority consensus.
[B3] D2 x I3 : 2 | reason=I3 requires structured probabilistic inputs tied to topology nodes, while D2 produces simple threshold flags, requiring a mapping layer between the two.
[B3] D2 x I4 : 2 | reason=Both operate on raw signal streams with overlapping responsibilities; risk of double-counting anomalies or conflicting thresholds requires clear role separation.
[B3] D3 x I1 : 2 | reason=I1 depends on transparent, quantitative comparison against a reference, while D3's black-box outputs lack the formal structure needed for clean residual comparison logic.
[B3] D3 x I2 : 3 | reason=Soft Voting Ensembles are explicitly documented as averaging black-box classifier outputs — directly designed for this pairing.
[B3] D3 x I3 : 3 | reason=Gao et al. documents the combination of RNN/deep learning with dependency modeling for temporal fault signature analysis.
[B3] D3 x I4 : 3 | reason=van Wyk et al. documents the explicit synergy between D3 (deep learning feature extraction) and I4 adaptive filtering.
[B3] D4 x I1 : 3 | reason=Active probe responses produce deterministic outputs that slot naturally into I1 comparison logic.
[B3] D4 x I2 : 2 | reason=Probes applied asymmetrically across voters could desynchronize consensus decisions; requires a probe-synchronization design decision.
[B3] D4 x I3 : 2 | reason=D4 probe-induced responses are transient events that don't match I3's continuous-dependency inference model, requiring event gating.
[B3] D4 x I4 : 2 | reason=D4 probes can corrupt I4's learned nominal baseline during or after warm-up, requiring explicit probe-aware filter masking.

**BLOCK B4 — I × R**

[B4] I1 x R1 : 3 | reason=I1 produces a conflict vector localizing the non-conforming sensor, which is exactly the input R1's fusion-list reconfiguration requires.
[B4] I1 x R2 : 3 | reason=Mahalanobis deviation magnitudes from I1 translate smoothly into graded weighting factors for R2.
[B4] I1 x R3 : 3 | reason=I1's hardware-vs-software fault localization directly identifies which branch to bypass.
[B4] I2 x R1 : 3 | reason=Mendonça et al. documents the explicit combination — voting identifies the Byzantine module and R1 excludes it.
[B4] I2 x R2 : 3 | reason=Voting reliability weights are inherently quantitative and feed R2's fusion weight redistribution naturally.
[B4] I2 x R3 : 3 | reason=When voting flags a specific voter as faulty, redirecting around that branch fits R3's bypass semantics cleanly.
[B4] I3 x R1 : 3 | reason=I3's failure-path localization pinpoints the faulty node, which R1 can then remove from fusion.
[B4] I3 x R2 : 3 | reason=I3's probability distribution over fault origins maps directly onto graded weight reductions in R2.
[B4] I3 x R3 : 3 | reason=Doc 15 documents I3 fault localization output directly informing the recovery module, a designed pairing.
[B4] I4 x R1 : 3 | reason=Per-channel deviation indicators from I4 are exactly the input R1 needs for targeted sensor exclusion.
[B4] I4 x R2 : 3 | reason=I4's dynamic health indicators and sensor-specific deviation thresholds map natively onto R2's adaptive weight vectors.
[B4] I4 x R3 : 3 | reason=I4 flags a specific faulty channel, which R3 can bypass via the redundant branch without conflict.

**BLOCK B5 — I × ID**

[B5] I1 x ID1 : 3 | reason=Doc 15 documents I1's interaction with rule-based identification via joint calibration and coordinate transformation.
[B5] I1 x ID2 : 3 | reason=I1's Mahalanobis innovation data is inherently statistical and flows natively into ID2's distribution-based tests.
[B5] I1 x ID3 : 3 | reason=I1 produces localized-fault signals that form a clean input boundary for ID3's supervised classification of fault type.
[B5] I2 x ID1 : 3 | reason=Voting consensus on which module is faulty feeds directly into rule-based fault-mode lookup with no intermediate translation.
[B5] I2 x ID2 : 3 | reason=Soft-voting probability distributions combine with ID2's statistical machinery (PCA, hypothesis testing) without architectural conflict.
[B5] I2 x ID3 : 3 | reason=Voter disagreement patterns provide informative features for ML classification of the fault's nature.
[B5] I3 x ID1 : 3 | reason=Rule engines can directly consume I3's failure-path localization as categorical input.
[B5] I3 x ID2 : 3 | reason=Bayesian probability distributions from I3 integrate seamlessly with ID2's statistical identification.
[B5] I3 x ID3 : 3 | reason=Gao et al. documents I3 + RNN combinations for temporal fault signature analysis — a designed pairing.
[B5] I4 x ID1 : 3 | reason=I4's dynamic thresholds and deviation indicators map onto ID1's predefined categorical fault rules.
[B5] I4 x ID2 : 3 | reason=Decoupled input disturbances from I4 feed naturally into ID2's statistical identification machinery.
[B5] I4 x ID3 : 3 | reason=I4's validated, sensor-specific deviation streams serve as clean feature inputs for ID3 classification.

**BLOCK B6 — ID × R**

[B6] ID1 x R1 : 3 | reason=Categorical fault types from ID1 drive deterministic exclusion decisions without translation overhead.
[B6] ID1 x R2 : 3 | reason=Severity levels produced by ID1 map cleanly onto tiered weighting in R2.
[B6] ID1 x R3 : 3 | reason=Doc 15 documents the dependency — ID1's categorical fault type triggers targeted bypass of the identified faulty sensor.
[B6] ID2 x R1 : 3 | reason=ID2's quantifiable deviation metrics inform exclusion thresholds without architectural friction.
[B6] ID2 x R2 : 3 | reason=Statistical deviation quantification from ID2 feeds R2's graded weighting especially naturally.
[B6] ID2 x R3 : 3 | reason=ID2's anomaly type/severity output cleanly triggers bypass when thresholds indicate hardware-branch failure.
[B6] ID3 x R1 : 3 | reason=ML Fault-Class outputs drive exclusion decisions directly — the explicit Fault Class is designed to invoke recovery.
[B6] ID3 x R2 : 3 | reason=ML confidence scores translate naturally into continuous weighting factors for R2.
[B6] ID3 x R3 : 3 | reason=Elhoseny/Lee document the explicit Fault Class invoking recovery including bypass, a natively supported pairing.
~~~

##### Gemini 2.0 Flash

~~~
LLM: Gemini 2.0 Flash

### Cross-Consistency Matrix Evaluation



**BLOCK B1 — D x R (12 cells)**
[B1] D1 x R1 : 3 | reason=Model-based residuals provide a clear binary trigger for the logical exclusion of a faulty sensor node.
[B1] D1 x R2 : 3 | reason=Continuous innovation residuals from Kalman filters are the standard input for calculating dynamic weighting factors.
[B1] D1 x R3 : 3 | reason=D1 residuals serve as the primary trigger for switching between redundant Kalman filter branches in the perception pipeline.
[B1] D2 x R1 : 3 | reason=Statistical signal outliers (e.g., Mahalanobis distance) allow for the rapid logical removal of corrupted data streams.
[B1] D2 x R2 : 3 | reason=Signal health scores derived from frequency analysis can directly scale the contribution of a sensor in the fusion layer.
[B1] D2 x R3 : 2 | reason=Signal-level detections may lack the state-aware context required to manage stable transitions between complex redundant filter banks.
[B1] D3 x R1 : 3 | reason=High reconstruction errors or OOD flags from deep learning models provide a reliable signal for sensor exclusion.
[B1] D3 x R2 : 3 | reason=Anomaly confidence scores from neural networks are ideal for implementation as continuous discounting factors in fusion.
[B1] D3 x R3 : 3 | reason=GAN-based confidence drops and OOD flags are documented recovery triggers for switching to alternative perception paths.
[B1] D4 x R1 : 3 | reason=Active verification confirms the persistence of a hardware fault before committing to permanent sensor exclusion.
[B1] D4 x R2 : 3 | reason=D4 is specifically documented as a method to verify if weight reconfiguration successfully restored the necessary system functionality.
[B1] D4 x R3 : 2 | reason=The timing overhead of active diagnostic probing may conflict with the hard real-time requirements for instantaneous hardware bypass switching.

**BLOCK B2 — D x ID (12 cells)**
[B2] D1 x ID1 : 3 | reason=Analytical residuals are easily mapped to expert-defined if-then gates for categorical fault classification.
[B2] D1 x ID2 : 3 | reason=Residual vectors provide the necessary structured input for statistical PCA or Mahalanobis-based identification.
[B2] D1 x ID3 : 3 | reason=Model-based residuals serve as high-signal features that improve the accuracy of downstream ML classifiers.
[B2] D2 x ID1 : 3 | reason=Signal-based flags such as dead zones or intensity spikes map directly to predefined expert diagnostic taxonomies.
[B2] D2 x ID2 : 3 | reason=Raw signal deviations are the fundamental input for distribution fitting and statistical hypothesis testing.
[B2] D2 x ID3 : 2 | reason=The high-dimensional nature of raw signal streams often requires significant pre-processing to be usable by supervised ML identifiers.
[B2] D3 x ID1 : 2 | reason=The "black-box" features extracted by deep learning detection are difficult to map onto human-interpretable expert rules.
[B2] D3 x ID2 : 3 | reason=Post-hoc statistics like Cosine Similarity or Mahalanobis distance over bottleneck layers enable statistical quantification of DL detections.
[B2] D3 x ID3 : 3 | reason=This represents a natural architectural progression where data-driven detection features are passed to a supervised classifier.
[B2] D4 x ID1 : 3 | reason=Deterministic responses provoked by active diagnosis allow for highly accurate rule-based categorization of fault modes.
[B2] D4 x ID2 : 2 | reason=The non-standard signal characteristics of active probes may complicate traditional statistical assumptions regarding nominal noise distributions.
[B2] D4 x ID3 : 2 | reason=ML identification requires specialized training data that includes the specific provoked responses generated by active diagnostic methods.

**BLOCK B3 — D x I (16 cells)**
[B3] D1 x I1 : 3 | reason=Residual outputs from mathematical observers serve as the primary baseline for cross-sensor output comparison logic.

[B3] D1 x I2 : 3 | reason=Model-based state estimates provide the high-fidelity independent inputs required for majority consensus voting.
[B3] D1 x I3 : 3 | reason=D1 provides the probabilistic residuals needed to trigger state transitions within dependency models like Petri nets.
[B3] D1 x I4 : 3 | reason=Residual outputs from model-based detection provide the baseline for training and updating individual adaptive filters.
[B3] D2 x I1 : 3 | reason=Signal-based anomaly detection is frequently paired with output comparison to isolate specific hardware-level discrepancies.
[B3] D2 x I2 : 3 | reason=Signal health indicators can be used to set the reliability weights required for Byzantine-tolerant voting protocols.
[B3] D2 x I3 : 2 | reason=Low-level signal noise patterns often lack the causal context needed to accurately traverse high-level dependency graphs.
[B3] D2 x I4 : 2 | reason=Raw signal noise can introduce instability during the mandatory warm-up periods required for adaptive filters to stabilize.
[B3] D3 x I1 : 3 | reason=ML-derived anomaly scores can be compared across heterogeneous modalities to isolate which sensor is non-conforming.
[B3] D3 x I2 : 3 | reason=Probabilities from data-driven classifiers are easily integrated into soft-voting ensembles for majority consensus.
[B3] D3 x I3 : 3 | reason=D3 provides the complex feature recognition necessary for temporal fault signature analysis within dependency models.
[B3] D3 x I4 : 3 | reason=Deep learning feature extraction synergizes with adaptive filtering to decouple environment shifts from hardware faults.
[B3] D4 x I1 : 3 | reason=Provoked responses from active diagnosis can be compared across redundant sensors to isolate the faulty hardware node.
[B3] D4 x I2 : 2 | reason=Active signal injection might disrupt the "uncompromised majority" assumption if not carefully coordinated across the voting population.
[B3] D4 x I3 : 3 | reason=D4 provides deterministic triggers that allow for the formal verification of reachability within dependency-based models.
[B3] D4 x I4 : 2 | reason=Periodic active probing can prevent adaptive filters from establishing a stable "nominal" behavior baseline for the sensor.

**BLOCK B4 — I x R (12 cells)**
[B4] I1 x R1 : 3 | reason=Comparison-based isolation directly identifies the non-conforming sensor node that must be logically excluded.
[B4] I1 x R2 : 3 | reason=Innovation residuals from output comparison provide the exact distance metric needed to calculate fusion discounting factors.
[B4] I1 x R3 : 3 | reason=I1 determines if a hardware discrepancy exists, serving as the binary switch for redundant branch masking.
[B4] I2 x R1 : 3 | reason=Voting results identify the specific Byzantine module that must be logically excluded to restore consensus.

[B4] I2 x R2 : 3 | reason=Consensus scores and majority weights can be directly applied as reliability factors in the fusion layer.
[B4] I2 x R3 : 2 | reason=The presence of a voting architecture usually implies active redundancy, which can make a bypass switch logically redundant.
[B4] I3 x R1 : 3 | reason=Causal mapping via Petri nets or Bayesian networks identifies the origin of a fault for targeted node exclusion.
[B4] I3 x R2 : 2 | reason=Dependency models often output discrete failure probabilities that are difficult to translate into nuanced, continuous weights.
[B4] I3 x R3 : 3 | reason=The localization output from dependency modeling is a documented direct trigger for recovery module routing decisions.
[B4] I4 x R1 : 3 | reason=When a sensor-specific health indicator from an adaptive filter drops, the node is logically removed from fusion.
[B4] I4 x R2 : 3 | reason=Trust factors derived from individual adaptive filters are the natural architectural input for weight reconfiguration.
[B4] I4 x R3 : 2 | reason=The inherent lag and warm-up requirements of adaptive filters may delay the instantaneous response needed for bypass recovery.

**BLOCK B5 — I x ID (12 cells)**
[B5] I1 x ID1 : 3 | reason=Discrepancy results from output comparison map directly to the formal agreement/discrepancy conditions in rule-based ID.
[B5] I1 x ID2 : 3 | reason=The innovation vectors generated during comparison are the standard data format for PCA-based statistical identification.
[B5] I1 x ID3 : 3 | reason=Isolation-based localization identifies the specific data region that the ML classifier must analyze for fault characterization.
[B5] I2 x ID1 : 3 | reason=Majority consensus outcomes are easily translated into categorical health status labels via expert rules.
[B5] I2 x ID2 : 2 | reason=Group-level voting statistics are less effective than individual residuals for the distribution fitting required in ID2.
[B5] I2 x ID3 : 3 | reason=Voter consensus patterns can serve as labels or confidence markers for supervised fault classification models.
[B5] I3 x ID1 : 3 | reason=States in a Petri net or Bayesian graph correspond directly to categorical failure mode taxonomies.
[B5] I3 x ID2 : 2 | reason=The probabilistic nature of causal graphs can conflict with the strict Gaussian assumptions often used in statistical identification.
[B5] I3 x ID3 : 3 | reason=Causal dependencies provide the temporal context required for effective recurrent neural network fault classification.
[B5] I4 x ID1 : 3 | reason=Health indicators and learned nominal behaviors provide the categorical data needed for expert system rules.
[B5] I4 x ID2 : 3 | reason=Validated measurements from adaptive filters allow ID2 to distinguish between gradual drift and abrupt anomalies more accurately.
[B5] I4 x ID3 : 3 | reason=Adaptive features provide a denoised and stable input stream that enhances the performance of ML identification.

**BLOCK B6 — ID x R (9 cells)**
[B6] ID1 x R1 : 3 | reason=Categorical identification of a critical fault via expert rules leads directly to logical sensor exclusion.
[B6] ID1 x R2 : 3 | reason=Rule-based severity levels allow for the implementation of discrete, multi-step weighting factors in the fusion layer.
[B6] ID1 x R3 : 3 | reason=Documented categorical fault types from rule-based identification can trigger targeted bypasses of specific faulty components.
[B6] ID2 x R1 : 3 | reason=Statistical quantification of fault severity (e.g., MD exceeding a threshold) provides a trigger for node exclusion.
[B6] ID2 x R2 : 3 | reason=Continuous statistical metrics, such as the Conflict factor K, are ideal for calculating precise fusion discounting factors.
[B6] ID2 x R3 : 2 | reason=The computational overhead of statistical analysis may prevent the "instant" response required for hardware branch switching.
[B6] ID3 x R1 : 3 | reason=Supervised classification of a fault with high confidence provides a reliable signal to logically remove the faulty node.
[B6] ID3 x R2 : 3 | reason=Confidence scores from neural network identifiers are the primary inputs for dynamic sensor weight reconfiguration.
[B6] ID3 x R3 : 3 | reason=The reclassification of an object into the "Fault Class" is a specific mechanism designed to invoke recovery interventions.
~~~

##### GPT-5.3

~~~
LLM: GPT-5.3

---

[B1] D1 x R1 : 3 | reason=Model-based residuals provide clear fault triggers enabling straightforward exclusion of faulty sensors.
[B1] D1 x R2 : 3 | reason=Residuals from D1 directly support dynamic weight adjustment in R2 without architectural conflict.
[B1] D1 x R3 : 3 | reason=D1 explicitly provides residual triggers for KF bank switching, making this pairing tightly integrated.

[B1] D2 x R1 : 3 | reason=Signal-level anomaly flags can directly trigger exclusion without additional architectural constraints.
[B1] D2 x R2 : 3 | reason=Signal health metrics can be used to adjust weights dynamically with no inherent conflict.
[B1] D2 x R3 : 2 | reason=D2 lacks structured residuals for precise switching decisions, requiring additional logic for reliable bypass triggering.

[B1] D3 x R1 : 3 | reason=Detected anomalies from D3 can directly initiate exclusion without incompatibility.
[B1] D3 x R2 : 3 | reason=Confidence scores from D3 integrate naturally into weight reconfiguration schemes.
[B1] D3 x R3 : 3 | reason=OOD or anomaly flags from D3 can trigger bypass decisions without architectural contradiction.

[B1] D4 x R1 : 2 | reason=Active probing introduces timing overhead that complicates immediate exclusion decisions under real-time constraints.
[B1] D4 x R2 : 2 | reason=Injected signals may interfere with stable weight estimation, requiring coordination mechanisms.
[B1] D4 x R3 : 2 | reason=Bypass decisions depend on probe cycles, introducing latency and synchronization constraints.

---

[B2] D1 x ID1 : 3 | reason=Residuals from D1 align well with rule-based thresholds for explicit fault classification.
[B2] D1 x ID2 : 3 | reason=Statistical identification naturally builds on residual distributions from D1.
[B2] D1 x ID3 : 3 | reason=Residuals provide strong features for ML classifiers without conflict.

[B2] D2 x ID1 : 3 | reason=Signal anomalies can be directly mapped to rule-based conditions.
[B2] D2 x ID2 : 3 | reason=Signal-derived statistics integrate naturally into statistical identification.
[B2] D2 x ID3 : 3 | reason=Raw signal features can feed ML classifiers without architectural issues.

[B2] D3 x ID1 : 2 | reason=Black-box detection outputs limit interpretability for rule-based classification, requiring feature translation.
[B2] D3 x ID2 : 3 | reason=Latent features from D3 can be statistically analyzed without conflict.
[B2] D3 x ID3 : 3 | reason=Both are ML-based and naturally compatible within a unified pipeline.

[B2] D4 x ID1 : 2 | reason=Probe-based outputs are not directly structured for rule-based classification without additional mapping.
[B2] D4 x ID2 : 2 | reason=Active diagnosis outputs require transformation before statistical analysis.
[B2] D4 x ID3 : 3 | reason=ML models can learn from probe-response patterns without inherent incompatibility.

---

[B3] D1 x I1 : 3 | reason=D1 residuals are explicitly required for output comparison, forming a strong synergy.
[B3] D1 x I2 : 3 | reason=Residual-based validation complements voting without conflict.
[B3] D1 x I3 : 3 | reason=Residuals provide inputs for probabilistic dependency models.
[B3] D1 x I4 : 3 | reason=Adaptive filters rely on D1 residuals for baseline learning.

[B3] D2 x I1 : 3 | reason=Signal anomalies can be compared across sensors for discrepancy detection.
[B3] D2 x I2 : 3 | reason=Signal-based outputs can be aggregated in voting schemes without conflict.
[B3] D2 x I3 : 2 | reason=Lack of structured probabilistic outputs requires additional modeling effort.
[B3] D2 x I4 : 3 | reason=Signal streams support adaptive filter learning directly.

[B3] D3 x I1 : 2 | reason=Black-box outputs complicate direct comparison across sensors without feature alignment.
[B3] D3 x I2 : 3 | reason=ML outputs can be incorporated into voting as probabilistic inputs.
[B3] D3 x I3 : 3 | reason=D3 integrates well with probabilistic dependency models and temporal learning.
[B3] D3 x I4 : 3 | reason=Adaptive filters can incorporate learned features from D3.

[B3] D4 x I1 : 2 | reason=Probe-induced responses complicate direct comparison with nominal sensor outputs.
[B3] D4 x I2 : 2 | reason=Active perturbations can distort consensus assumptions in voting.
[B3] D4 x I3 : 3 | reason=Probe responses can enrich dependency models without contradiction.
[B3] D4 x I4 : 2 | reason=Injected signals interfere with stable adaptive learning processes.

---

[B4] I1 x R1 : 3 | reason=I1 provides precise fault localization enabling direct exclusion.
[B4] I1 x R2 : 3 | reason=Comparison outputs directly inform weight adjustments.
[B4] I1 x R3 : 3 | reason=Localized discrepancies allow targeted bypass switching.

[B4] I2 x R1 : 3 | reason=Voting identifies faulty modules for exclusion effectively.
[B4] I2 x R2 : 3 | reason=Voting confidence can be translated into weight adjustments.
[B4] I2 x R3 : 3 | reason=Consensus failure can trigger bypass of faulty branches.

[B4] I3 x R1 : 2 | reason=Probabilistic outputs may delay decisive exclusion under real-time constraints.
[B4] I3 x R2 : 3 | reason=Fault probabilities integrate well into weight adaptation schemes.
[B4] I3 x R3 : 3 | reason=Dependency modeling directly informs bypass routing decisions.

[B4] I4 x R1 : 2 | reason=Gradual trust adaptation may delay hard exclusion decisions.
[B4] I4 x R2 : 3 | reason=Adaptive trust factors naturally map to weight reconfiguration.
[B4] I4 x R3 : 2 | reason=Lack of discrete fault localization complicates immediate bypass switching.

---

[B5] I1 x ID1 : 3 | reason=Comparison outputs align directly with rule-based classification logic.
[B5] I1 x ID2 : 3 | reason=Residual discrepancies can be statistically analyzed without conflict.
[B5] I1 x ID3 : 3 | reason=Comparison features serve as valid inputs for ML classifiers.

[B5] I2 x ID1 : 3 | reason=Voting outcomes can be encoded into explicit rule-based decisions.
[B5] I2 x ID2 : 3 | reason=Consensus outputs can be statistically interpreted.
[B5] I2 x ID3 : 3 | reason=Voting results can feed ML classifiers as features.

[B5] I3 x ID1 : 2 | reason=Complex probabilistic outputs require simplification for rule-based systems.
[B5] I3 x ID2 : 3 | reason=Statistical identification aligns well with probabilistic outputs.
[B5] I3 x ID3 : 3 | reason=ML models can directly consume probabilistic features.

[B5] I4 x ID1 : 2 | reason=Adaptive outputs lack discrete thresholds for rule-based classification.
[B5] I4 x ID2 : 3 | reason=Adaptive filter outputs can be statistically analyzed.
[B5] I4 x ID3 : 3 | reason=Adaptive features integrate well into ML classifiers.

---

[B6] ID1 x R1 : 3 | reason=Explicit fault categories directly trigger exclusion logic.
[B6] ID1 x R2 : 3 | reason=Rule-based severity can guide weight adjustment decisions.
[B6] ID1 x R3 : 3 | reason=Categorical fault identification enables targeted bypass.

[B6] ID2 x R1 : 3 | reason=Statistical thresholds can trigger exclusion decisions.
[B6] ID2 x R2 : 3 | reason=Statistical metrics map naturally to weight scaling.
[B6] ID2 x R3 : 2 | reason=Probabilistic outputs may lack discrete triggers for immediate bypass.

[B6] ID3 x R1 : 3 | reason=ML classification outputs can directly indicate faulty components for exclusion.
[B6] ID3 x R2 : 3 | reason=Confidence scores integrate naturally into weight adjustment.
[B6] ID3 x R3 : 3 | reason=Fault classification outputs can trigger bypass decisions without conflict.
~~~

##### DeepSeek-R1

~~~
LLM: DeepSeek-R1

[B1] D1 x R1 : 3 | reason=Model-based residuals provide clear fault localization that directly informs sensor exclusion without any architectural conflict.
[B1] D1 x R2 : 3 | reason=D1 residual magnitudes naturally translate into weakening factors for weight reconfiguration, as documented in belief-function discounting synergies.
[B1] D1 x R3 : 3 | reason=D1 residual triggers are the primary mechanism for KF bank switching, enabling instantaneous bypass actuation.
[B1] D2 x R1 : 3 | reason=Signal-based health flags pinpoint the faulty sensor, allowing exclusion without requiring additional translation logic.
[B1] D2 x R2 : 3 | reason=Signal health metrics can directly modulate fusion weights, with no structural incompatibility between the two.
[B1] D2 x R3 : 3 | reason=Signal-based anomaly identification cleanly triggers redundant path switching when a flagged sensor has an alternative.
[B1] D3 x R1 : 3 | reason=Data-driven anomaly scores localize faults to a sensor, enabling exclusion; the black-box nature does not create a pairwise conflict.
[B1] D3 x R2 : 3 | reason=Confidence scores from D3 can be mapped to discounting weights without contradicting the recovery logic.
[B1] D3 x R3 : 3 | reason=GAN-based OOD flags can directly trigger bypass switching, with no inherent architectural mismatch.
[B1] D4 x R1 : 3 | reason=Active probing conclusively identifies a faulty sensor, which can then be excluded; the probe–exclude sequence has no logical contradiction.
[B1] D4 x R2 : 3 | reason=D4 is explicitly documented as a verification step for weight reconfiguration, demonstrating full synergy.
[B1] D4 x R3 : 3 | reason=Probe results can initiate bypass to a verified healthy branch, without conflict in the perception pipeline.

[B2] D1 x ID1 : 3 | reason=Deterministic residuals directly satisfy rule-based conditions like the Bader dual-sensor agreement/discrepancy check.
[B2] D1 x ID2 : 3 | reason=Innovation sequences and conflict factor K serve as ideal inputs for statistical distance and PCA-based classification.
[B2] D1 x ID3 : 3 | reason=Model-based residuals are documented as high-signal features for downstream supervised classifiers, ensuring compatibility.
[B2] D2 x ID1 : 3 | reason=Signal health indicators and anomaly flags can be evaluated by explicit if-then rules without translation barriers.
[B2] D2 x ID2 : 3 | reason=Frequency-domain scores and Mahalanobis distances from D2 feed directly into statistical identification routines.
[B2] D2 x ID3 : 3 | reason=Wavelet coefficients and PSD metrics can serve as structured features for an ML classifier with no inherent mismatch.
[B2] D3 x ID1 : 3 | reason=Anomaly scores and reconstruction errors can be thresholded by rule-based logic; the opacity of D3 does not prevent this mapping.
[B2] D3 x ID2 : 3 | reason=Latent-space representations from D3 are explicitly used with cosine similarity and Mahalanobis distance for statistical identification.
[B2] D3 x ID3 : 3 | reason=Feature-space outputs from one network can be classified by another; stacking does not introduce a pairwise incompatibility.
[B2] D4 x ID1 : 3 | reason=Probe response signatures can be checked against predefined fault patterns using rule-based conditions without conflict.
[B2] D4 x ID2 : 3 | reason=Injection–response data forms a structured measurement set suitable for hypothesis testing and PCA-based identification.
[B2] D4 x ID3 : 3 | reason=Probe responses can be engineered as features for an ML classifier, posing no architectural contradiction.

[B3] D1 x I1 : 3 | reason=Residual outputs serve as the primary baseline for output comparison, creating a tightly coupled and documented synergy.
[B3] D1 x I2 : 3 | reason=Per-sensor residual checks can feed into a voting scheme without architectural restriction.
[B3] D1 x I3 : 3 | reason=Model-based residuals provide quantitative evidence for Bayesian/Petri net propagation analysis, fully compatible.
[B3] D1 x I4 : 3 | reason=Residual signals directly supply the baseline for adaptive filter training and discounting factor computation.
[B3] D2 x I1 : 3 | reason=Signal-based fault flags can trigger a targeted output comparison among sensors, with no design contradiction.
[B3] D2 x I2 : 3 | reason=Sensor-specific health metrics can be aggregated by voting logic without requiring structural changes.
[B3] D2 x I3 : 3 | reason=Frequency-domain anomalies can serve as input nodes in a dependency graph, compatible with the modeling paradigm.
[B3] D2 x I4 : 3 | reason=Signal health streams and adaptive filter learning operate on the same channel data without mutual interference.
[B3] D3 x I1 : 3 | reason=Data-driven anomaly detection can initiate isolation via output comparison, as both methods operate on sensor measurements.
[B3] D3 x I2 : 3 | reason=Deep anomaly scores per sensor can be cast as votes in a consensus framework with no inherent conflict.
[B3] D3 x I3 : 3 | reason=Probabilistic anomaly outputs can inform Bayesian network priors, maintaining logical compatibility.
[B3] D3 x I4 : 3 | reason=Anomaly features from D3 can be used to adjust adaptive filter baselines; sequential operation is architecturally sound.
[B3] D4 x I1 : 3 | reason=Active probing can provoke deterministic responses that output comparison then evaluates, with no pairwise contradiction.
[B3] D4 x I2 : 3 | reason=Probe results per sensor can be aggregated in a voting scheme to isolate the faulty node.
[B3] D4 x I3 : 3 | reason=Injection–response traces can be modeled as causal evidence in dependency graphs without conflict.
[B3] D4 x I4 : 2 | reason=Active signal injection introduces deliberate non-nominal perturbations that corrupt the learned baseline of I4 adaptive filters, requiring explicit coordination to pause adaptation during probing.

[B4] I1 x R1 : 3 | reason=Output comparison directly identifies the non-conforming sensor, which exclusion then cleanly removes from the pipeline.
[B4] I1 x R2 : 3 | reason=Innovation magnitudes from I1 are the standard input for computing weakening factors in weight reconfiguration.
[B4] I1 x R3 : 3 | reason=Isolation of a specific faulty sensor branch allows bypass switching to a redundant path without any architectural gap.
[B4] I2 x R1 : 3 | reason=Voting explicitly identifies the Byzantine module, enabling targeted exclusion as documented in Mendonça et al.
[B4] I2 x R2 : 3 | reason=Vote confidence levels can be directly transformed into sensor weights, with no structural conflict.
[B4] I2 x R3 : 3 | reason=Consensus isolation selects the faulty sensor, and bypass reroutes to a healthy redundant member if available.
[B4] I3 x R1 : 3 | reason=Fault probability distributions from dependency modeling pinpoint the sensor to exclude, seamlessly feeding into R1.
[B4] I3 x R2 : 3 | reason=Node failure probabilities naturally serve as discounting factors, aligning perfectly with weight reconfiguration.
[B4] I3 x R3 : 3 | reason=I3 fault localization output directly informs the recovery module, as documented, enabling targeted bypass.
[B4] I4 x R1 : 3 | reason=Dynamic health indicators below a threshold can trigger exclusion, with no logical incompatibility.
[B4] I4 x R2 : 3 | reason=Weakening factors p_trust from I4 are exactly the weights used by R2; the two form a direct synergy.
[B4] I4 x R3 : 3 | reason=Adaptive filter health assessments can flag a sensor for bypass without conflicting with the switching mechanism.

[B5] I1 x ID1 : 3 | reason=The isolated sensor identity and innovation magnitude provide clear inputs for rule-based fault classification.
[B5] I1 x ID2 : 3 | reason=Innovation sequences and conflict vectors are well-suited for statistical distance and PCA-based identification.
[B5] I1 x ID3 : 3 | reason=Isolation features can be fed into an ML classifier without imposing additional architectural constraints.
[B5] I2 x ID1 : 3 | reason=Voting results and the identity of the outvoted module can be evaluated by explicit fault-signature rules.
[B5] I2 x ID2 : 3 | reason=Consensus margins and voting patterns provide quantifiable metrics for statistical fault-type inference.
[B5] I2 x ID3 : 3 | reason=Voting outcomes can serve as structured features for a supervised classifier, fully compatible.
[B5] I3 x ID1 : 3 | reason=Probabilistic fault origin outputs can be thresholded by rule-based logic to classify fault type.
[B5] I3 x ID2 : 3 | reason=Dependency-model probabilities form a natural input for statistical analysis and hypothesis testing.
[B5] I3 x ID3 : 3 | reason=Bayesian network posteriors can be encoded as features for an ML identification stage without conflict.
[B5] I4 x ID1 : 3 | reason=Sensor-specific health indicators and learned deviation patterns can be checked against fault taxonomy rules.
[B5] I4 x ID2 : 3 | reason=Dynamic health metrics and adaptive thresholds provide continuous data for statistical classification routines.
[B5] I4 x ID3 : 3 | reason=Health indicators and weakening factors can be input to a neural classifier; no architectural mismatch exists.

[B6] ID1 x R1 : 3 | reason=A categorical fault type can trigger targeted exclusion of the identified sensor, with no logical gap.
[B6] ID1 x R2 : 3 | reason=Rule-based severity classification can determine the degree of weight reduction, aligning cleanly with R2.
[B6] ID1 x R3 : 3 | reason=ID1 fault categorization can directly invoke a specific bypass path, as documented in the context.
[B6] ID2 x R1 : 3 | reason=Statistical severity metrics can inform the exclusion decision without introducing any design conflict.
[B6] ID2 x R2 : 3 | reason=Deviation magnitudes from ID2 can be mapped to discounting factors, fully compatible with weight reconfiguration.
[B6] ID2 x R3 : 3 | reason=Statistical fault identification can trigger bypass when deviation exceeds a threshold, with no architectural mismatch.
[B6] ID3 x R1 : 3 | reason=ML-classified fault types can be used to decide sensor exclusion, maintaining pipeline consistency.
[B6] ID3 x R2 : 3 | reason=Classifier confidence scores can be transformed into fusion weights without contradicting the recovery approach.
[B6] ID3 x R3 : 3 | reason=ML identification of a fault class can initiate a targeted redundant-branch switch with no pairwise conflict.
~~~

#### Vetor B — CCM

In [108]:
CCM = {
    'Claude': {
        # B1: D x R
        ('D1','R1'):3, ('D1','R2'):3, ('D1','R3'):3,
        ('D2','R1'):3, ('D2','R2'):2, ('D2','R3'):3,
        ('D3','R1'):3, ('D3','R2'):3, ('D3','R3'):3,
        ('D4','R1'):2, ('D4','R2'):3, ('D4','R3'):3,
        # B2: D x ID
        ('D1','ID1'):3, ('D1','ID2'):3, ('D1','ID3'):3,
        ('D2','ID1'):3, ('D2','ID2'):3, ('D2','ID3'):2,
        ('D3','ID1'):2, ('D3','ID2'):3, ('D3','ID3'):3,
        ('D4','ID1'):3, ('D4','ID2'):3, ('D4','ID3'):2,
        # B3: D x I
        ('D1','I1'):3, ('D1','I2'):3, ('D1','I3'):3, ('D1','I4'):3,
        ('D2','I1'):3, ('D2','I2'):3, ('D2','I3'):2, ('D2','I4'):2,
        ('D3','I1'):2, ('D3','I2'):3, ('D3','I3'):3, ('D3','I4'):3,
        ('D4','I1'):3, ('D4','I2'):2, ('D4','I3'):2, ('D4','I4'):2,
        # B4: I x R
        ('I1','R1'):3, ('I1','R2'):3, ('I1','R3'):3,
        ('I2','R1'):3, ('I2','R2'):3, ('I2','R3'):3,
        ('I3','R1'):3, ('I3','R2'):3, ('I3','R3'):3,
        ('I4','R1'):3, ('I4','R2'):3, ('I4','R3'):3,
        # B5: I x ID
        ('I1','ID1'):3, ('I1','ID2'):3, ('I1','ID3'):3,
        ('I2','ID1'):3, ('I2','ID2'):3, ('I2','ID3'):3,
        ('I3','ID1'):3, ('I3','ID2'):3, ('I3','ID3'):3,
        ('I4','ID1'):3, ('I4','ID2'):3, ('I4','ID3'):3,
        # B6: ID x R
        ('ID1','R1'):3, ('ID1','R2'):3, ('ID1','R3'):3,
        ('ID2','R1'):3, ('ID2','R2'):3, ('ID2','R3'):3,
        ('ID3','R1'):3, ('ID3','R2'):3, ('ID3','R3'):3,
    },
    'Gemini': {
        # B1: D x R
        ('D1','R1'):3, ('D1','R2'):3, ('D1','R3'):3,
        ('D2','R1'):3, ('D2','R2'):3, ('D2','R3'):2,
        ('D3','R1'):3, ('D3','R2'):3, ('D3','R3'):3,
        ('D4','R1'):3, ('D4','R2'):3, ('D4','R3'):2,
        # B2: D x ID
        ('D1','ID1'):3, ('D1','ID2'):3, ('D1','ID3'):3,
        ('D2','ID1'):3, ('D2','ID2'):3, ('D2','ID3'):2,
        ('D3','ID1'):2, ('D3','ID2'):3, ('D3','ID3'):3,
        ('D4','ID1'):3, ('D4','ID2'):2, ('D4','ID3'):2,
        # B3: D x I
        ('D1','I1'):3, ('D1','I2'):3, ('D1','I3'):3, ('D1','I4'):3,
        ('D2','I1'):3, ('D2','I2'):3, ('D2','I3'):2, ('D2','I4'):2,
        ('D3','I1'):3, ('D3','I2'):3, ('D3','I3'):3, ('D3','I4'):3,
        ('D4','I1'):3, ('D4','I2'):2, ('D4','I3'):3, ('D4','I4'):2,
        # B4: I x R
        ('I1','R1'):3, ('I1','R2'):3, ('I1','R3'):3,
        ('I2','R1'):3, ('I2','R2'):3, ('I2','R3'):2,
        ('I3','R1'):3, ('I3','R2'):2, ('I3','R3'):3,
        ('I4','R1'):3, ('I4','R2'):3, ('I4','R3'):2,
        # B5: I x ID
        ('I1','ID1'):3, ('I1','ID2'):3, ('I1','ID3'):3,
        ('I2','ID1'):3, ('I2','ID2'):2, ('I2','ID3'):3,
        ('I3','ID1'):3, ('I3','ID2'):2, ('I3','ID3'):3,
        ('I4','ID1'):3, ('I4','ID2'):3, ('I4','ID3'):3,
        # B6: ID x R
        ('ID1','R1'):3, ('ID1','R2'):3, ('ID1','R3'):3,
        ('ID2','R1'):3, ('ID2','R2'):3, ('ID2','R3'):2,
        ('ID3','R1'):3, ('ID3','R2'):3, ('ID3','R3'):3,
    },
    'GPT': {
        # B1: D x R
        ('D1','R1'):3, ('D1','R2'):3, ('D1','R3'):3,
        ('D2','R1'):3, ('D2','R2'):3, ('D2','R3'):2,
        ('D3','R1'):3, ('D3','R2'):3, ('D3','R3'):3,
        ('D4','R1'):2, ('D4','R2'):2, ('D4','R3'):2,
        # B2: D x ID
        ('D1','ID1'):3, ('D1','ID2'):3, ('D1','ID3'):3,
        ('D2','ID1'):3, ('D2','ID2'):3, ('D2','ID3'):3,
        ('D3','ID1'):2, ('D3','ID2'):3, ('D3','ID3'):3,
        ('D4','ID1'):2, ('D4','ID2'):2, ('D4','ID3'):3,
        # B3: D x I
        ('D1','I1'):3, ('D1','I2'):3, ('D1','I3'):3, ('D1','I4'):3,
        ('D2','I1'):3, ('D2','I2'):3, ('D2','I3'):2, ('D2','I4'):3,
        ('D3','I1'):2, ('D3','I2'):3, ('D3','I3'):3, ('D3','I4'):3,
        ('D4','I1'):2, ('D4','I2'):2, ('D4','I3'):3, ('D4','I4'):2,
        # B4: I x R
        ('I1','R1'):3, ('I1','R2'):3, ('I1','R3'):3,
        ('I2','R1'):3, ('I2','R2'):3, ('I2','R3'):3,
        ('I3','R1'):2, ('I3','R2'):3, ('I3','R3'):3,
        ('I4','R1'):2, ('I4','R2'):3, ('I4','R3'):2,
        # B5: I x ID
        ('I1','ID1'):3, ('I1','ID2'):3, ('I1','ID3'):3,
        ('I2','ID1'):3, ('I2','ID2'):3, ('I2','ID3'):3,
        ('I3','ID1'):2, ('I3','ID2'):3, ('I3','ID3'):3,
        ('I4','ID1'):2, ('I4','ID2'):3, ('I4','ID3'):3,
        # B6: ID x R
        ('ID1','R1'):3, ('ID1','R2'):3, ('ID1','R3'):3,
        ('ID2','R1'):3, ('ID2','R2'):3, ('ID2','R3'):2,
        ('ID3','R1'):3, ('ID3','R2'):3, ('ID3','R3'):3,
    },
    'DeepSeek': {
        # B1: D x R
        ('D1','R1'):3, ('D1','R2'):3, ('D1','R3'):3,
        ('D2','R1'):3, ('D2','R2'):3, ('D2','R3'):3,
        ('D3','R1'):3, ('D3','R2'):3, ('D3','R3'):3,
        ('D4','R1'):3, ('D4','R2'):3, ('D4','R3'):3,
        # B2: D x ID
        ('D1','ID1'):3, ('D1','ID2'):3, ('D1','ID3'):3,
        ('D2','ID1'):3, ('D2','ID2'):3, ('D2','ID3'):3,
        ('D3','ID1'):3, ('D3','ID2'):3, ('D3','ID3'):3,
        ('D4','ID1'):3, ('D4','ID2'):3, ('D4','ID3'):3,
        # B3: D x I
        ('D1','I1'):3, ('D1','I2'):3, ('D1','I3'):3, ('D1','I4'):3,
        ('D2','I1'):3, ('D2','I2'):3, ('D2','I3'):3, ('D2','I4'):3,
        ('D3','I1'):3, ('D3','I2'):3, ('D3','I3'):3, ('D3','I4'):3,
        ('D4','I1'):3, ('D4','I2'):3, ('D4','I3'):3, ('D4','I4'):2,
        # B4: I x R
        ('I1','R1'):3, ('I1','R2'):3, ('I1','R3'):3,
        ('I2','R1'):3, ('I2','R2'):3, ('I2','R3'):3,
        ('I3','R1'):3, ('I3','R2'):3, ('I3','R3'):3,
        ('I4','R1'):3, ('I4','R2'):3, ('I4','R3'):3,
        # B5: I x ID
        ('I1','ID1'):3, ('I1','ID2'):3, ('I1','ID3'):3,
        ('I2','ID1'):3, ('I2','ID2'):3, ('I2','ID3'):3,
        ('I3','ID1'):3, ('I3','ID2'):3, ('I3','ID3'):3,
        ('I4','ID1'):3, ('I4','ID2'):3, ('I4','ID3'):3,
        # B6: ID x R
        ('ID1','R1'):3, ('ID1','R2'):3, ('ID1','R3'):3,
        ('ID2','R1'):3, ('ID2','R2'):3, ('ID2','R3'):3,
        ('ID3','R1'):3, ('ID3','R2'):3, ('ID3','R3'):3,
    }
}

# Definir todos os pares na ordem dos blocos
PAIRS_ORDERED = [
    # B1
    ('D1','R1'), ('D1','R2'), ('D1','R3'),
    ('D2','R1'), ('D2','R2'), ('D2','R3'),
    ('D3','R1'), ('D3','R2'), ('D3','R3'),
    ('D4','R1'), ('D4','R2'), ('D4','R3'),
    # B2
    ('D1','ID1'), ('D1','ID2'), ('D1','ID3'),
    ('D2','ID1'), ('D2','ID2'), ('D2','ID3'),
    ('D3','ID1'), ('D3','ID2'), ('D3','ID3'),
    ('D4','ID1'), ('D4','ID2'), ('D4','ID3'),
    # B3
    ('D1','I1'), ('D1','I2'), ('D1','I3'), ('D1','I4'),
    ('D2','I1'), ('D2','I2'), ('D2','I3'), ('D2','I4'),
    ('D3','I1'), ('D3','I2'), ('D3','I3'), ('D3','I4'),
    ('D4','I1'), ('D4','I2'), ('D4','I3'), ('D4','I4'),
    # B4
    ('I1','R1'), ('I1','R2'), ('I1','R3'),
    ('I2','R1'), ('I2','R2'), ('I2','R3'),
    ('I3','R1'), ('I3','R2'), ('I3','R3'),
    ('I4','R1'), ('I4','R2'), ('I4','R3'),
    # B5
    ('I1','ID1'), ('I1','ID2'), ('I1','ID3'),
    ('I2','ID1'), ('I2','ID2'), ('I2','ID3'),
    ('I3','ID1'), ('I3','ID2'), ('I3','ID3'),
    ('I4','ID1'), ('I4','ID2'), ('I4','ID3'),
    # B6
    ('ID1','R1'), ('ID1','R2'), ('ID1','R3'),
    ('ID2','R1'), ('ID2','R2'), ('ID2','R3'),
    ('ID3','R1'), ('ID3','R2'), ('ID3','R3'),
]

llm_names = list(CCM.keys())

# Calcular score final e divergencias
records = []
for pair in PAIRS_ORDERED:
    scores = np.array([CCM[llm][pair] for llm in llm_names])
    mean = scores.mean()
    std  = scores.std()
    final = int(np.round(mean))
    records.append({
        'pair': f"{pair[0]}x{pair[1]}",
        'Claude': scores[0], 'Gemini': scores[1],
        'GPT': scores[2], 'DeepSeek': scores[3],
        'mean': round(mean, 2), 'std': round(std, 2),
        'final': final
    })

df_ccm = pd.DataFrame(records).set_index('pair')

# Sumario
n_total = len(df_ccm)
n_3 = (df_ccm['final'] == 3).sum()
n_2 = (df_ccm['final'] == 2).sum()
n_1 = (df_ccm['final'] == 1).sum()
n_0 = (df_ccm['final'] == 0).sum()

print("CCM Consolidada -- Sumario")
print("-" * 40)
print(f"Total de celulas:  {n_total}")
print(f"  Score 3:  {n_3} ({100*n_3/n_total:.1f}%)")
print(f"  Score 2:  {n_2} ({100*n_2/n_total:.1f}%)")
print(f"  Score 1:  {n_1} ({100*n_1/n_total:.1f}%)")
print(f"  Score 0:  {n_0} ({100*n_0/n_total:.1f}%)")

# Celulas com divergencia
print("\nCelulas com divergencia (std > 0.5):")
div = df_ccm[df_ccm['std'] > 0.5]
print(f"{'Par':<12} {'Claude':>7} {'Gemini':>7} {'GPT':>7} "
      f"{'DeepSeek':>9} {'Media':>7} {'Std':>6} {'Final':>6}")
print("-" * 65)
for idx, row in div.iterrows():
    print(f"{idx:<12} {row['Claude']:>7} {row['Gemini']:>7} "
          f"{row['GPT']:>7} {row['DeepSeek']:>9} "
          f"{row['mean']:>7.2f} {row['std']:>6.2f} {row['final']:>6}")

CCM_SYM = {}
for pair_str, row in df_ccm.iterrows():
    a, b = pair_str.split('x')
    v = row['final']
    CCM_SYM[(a, b)] = v
    CCM_SYM[(b, a)] = v

def get_ccm(da1, da2):
    """
    Retorna score CCM consolidado para um par de DAs.
    Escala {0, 1, 2, 3}: 3=plenamente compativel, 2=restricoes moderadas,
    1=restricoes severas, 0=incompativel.
    """
    return CCM_SYM.get((da1, da2), CCM_SYM.get((da2, da1), None))

# Verificacao
n_total = len(CCM_SYM) // 2
counts = {s: sum(1 for v in CCM_SYM.values() if v == s) // 2 for s in [3, 2, 1, 0]}
print(f"CCM_SYM carregada: {n_total} pares")
for s, n in counts.items():
    print(f"  Score {s}: {n} ({100*n/n_total:.1f}%)")

CCM Consolidada -- Sumario
----------------------------------------
Total de celulas:  73
  Score 3:  59 (80.8%)
  Score 2:  14 (19.2%)
  Score 1:  0 (0.0%)
  Score 0:  0 (0.0%)

Celulas com divergencia (std > 0.5):
Par           Claude  Gemini     GPT  DeepSeek   Media    Std  Final
-----------------------------------------------------------------
CCM_SYM carregada: 73 pares
  Score 3: 59 (80.8%)
  Score 2: 14 (19.2%)
  Score 1: 0 (0.0%)
  Score 0: 0 (0.0%)


### Cenários SC1–SC4
 
Pesos de módulo homogêneos em todos os cenários (w_D = w_I = w_ID = w_R = 0.25).  
Diferenciação exclusivamente pelos pesos dos critérios (w_C1–w_C5).

In [109]:
# Cenários: pesos dos módulos e dos critérios
SCENARIOS = {
    'SC1': {
        'name': 'Uniforme',
        'module_weights': {'D': .25, 'I': .25, 'ID': .25, 'R': .25},
        'criteria_weights': {'C1': .20, 'C2': .20, 'C3': .20, 'C4': .20, 'C5': .20}
    },
    'SC2': {
        'name': 'Segurança / certificação',
        'module_weights': {'D': .25, 'I': .25, 'ID': .25, 'R': .25},
        'criteria_weights': {'C1': .35, 'C2': .10, 'C3': .10, 'C4': .35, 'C5': .10}
    },
    'SC3': {
        'name': 'Cobertura / desempenho',
        'module_weights': {'D': .25, 'I': .25, 'ID': .25, 'R': .25},
        'criteria_weights': {'C1': .10, 'C2': .20, 'C3': .20, 'C4': .10, 'C5': .40}
    },
    'SC4': {
        'name': 'Eficiência de recursos',
        'module_weights': {'D': .25, 'I': .25, 'ID': .25, 'R': .25},
        'criteria_weights': {'C1': .10, 'C2': .35, 'C3': .35, 'C4': .10, 'C5': .10}
    }
}

# Verificação: pesos dos critérios somam 1.0
for sc_id, sc in SCENARIOS.items():
    total = sum(sc['criteria_weights'].values())
    assert abs(total - 1.0) < 1e-9, f"{sc_id}: pesos dos critérios somam {total}"

# Exibição
rows = []
for sc_id, sc in SCENARIOS.items():
    row = {'Cenário': sc_id, 'Semântica': sc['name']}
    row.update({f'w_{k}': v for k, v in sc['module_weights'].items()})
    row.update({f'w_{k}': v for k, v in sc['criteria_weights'].items()})
    rows.append(row)

df_scenarios = pd.DataFrame(rows).set_index('Cenário')
print("Cenários SC1-SC4")
print(df_scenarios.to_string())

Cenários SC1-SC4
                        Semântica   w_D   w_I  w_ID   w_R  w_C1  w_C2  w_C3  w_C4  w_C5
Cenário                                                                                
SC1                      Uniforme 0.250 0.250 0.250 0.250 0.200 0.200 0.200 0.200 0.200
SC2      Segurança / certificação 0.250 0.250 0.250 0.250 0.350 0.100 0.100 0.350 0.100
SC3        Cobertura / desempenho 0.250 0.250 0.250 0.250 0.100 0.200 0.200 0.100 0.400
SC4        Eficiência de recursos 0.250 0.250 0.250 0.250 0.100 0.350 0.350 0.100 0.100


## Espaco de Solucoes


### Geracao do produto cartesiano D x I x ID x R


In [110]:
def generate_configurations():
    """
    Gera todas as 144 configuracoes do produto cartesiano D x I x ID x R.
    Retorna lista de dicts: {'D': da_d, 'I': da_i, 'ID': da_id, 'R': da_r}
    """
    configs = []
    for d, i, id_, r in itertools.product(
        MODULE_VALUES['D'],
        MODULE_VALUES['I'],
        MODULE_VALUES['ID'],
        MODULE_VALUES['R']
    ):
        configs.append({'D': d, 'I': i, 'ID': id_, 'R': r})
    return configs


def compute_w(config):
    """
    Calcula w(S) = minimo das compatibilidades par-a-par entre DAs
    de modulos diferentes na configuracao S.
    Fonte: Levin (2012), Secao 3.7 -- N(S) = (w(S); n(S))
    Os 6 pares correspondem aos blocos B1-B6 da CCM E5.
    """
    das = [config['D'], config['I'], config['ID'], config['R']]
    scores = []
    for da1, da2 in itertools.combinations(das, 2):
        score = get_ccm(da1, da2)
        if score is None:
            raise ValueError(f"Par nao encontrado na CCM: ({da1}, {da2})")
        scores.append(score)
    return min(scores)


# Gerar todas as configuracoes com w(S)
all_configs = generate_configurations()
for cfg in all_configs:
    cfg['w'] = compute_w(cfg)

print(f"Configuracoes geradas: {len(all_configs)}")
print(f"  w(S) = 3: {sum(1 for c in all_configs if c['w'] == 3)}")
print(f"  w(S) = 2: {sum(1 for c in all_configs if c['w'] == 2)}")
print(f"  w(S) = 1: {sum(1 for c in all_configs if c['w'] == 1)}")
print(f"  w(S) = 0: {sum(1 for c in all_configs if c['w'] == 0)}")

Configuracoes geradas: 144
  w(S) = 3: 54
  w(S) = 2: 90
  w(S) = 1: 0
  w(S) = 0: 0


### Filtragem do conjunto admissivel

Criterio de admissibilidade: w(S) >= 0.5.

CCM tem zero celulas em 0.0 -- todas as configuracoes tem pelo menos compatibilidade condicional em todos os pares.

O threshold w >= 0.5 admite o espaco completo excluindo apenas incompatibilidades absolutas.

In [111]:
def filter_admissible(configs, threshold=2):
    admissible = [c for c in configs if c['w'] >= threshold]
    for i, cfg in enumerate(admissible, 2):
        cfg['id'] = f"S{(i-1):02d}"
    return admissible

for cfg in all_configs:
    cfg.pop('id', None)

admissible = filter_admissible(all_configs, threshold=1)

print(f"Espaco admissivel: {len(admissible)} de {len(all_configs)} configuracoes")
print(f"Reducao: {100*(1 - len(admissible)/len(all_configs)):.1f}%")
print(f"  w(S) = 3: {sum(1 for c in admissible if c['w'] == 3)} configuracoes")
print(f"  w(S) = 2: {sum(1 for c in admissible if c['w'] == 2)} configuracoes")
print()

# Tabela do espaco admissivel
df_admissible = pd.DataFrame([
    {
        'ID': c['id'],
        'D': f"{c['D']} ({DA_NAMES[c['D']]})",
        'I': f"{c['I']} ({DA_NAMES[c['I']]})",
        'ID_': f"{c['ID']} ({DA_NAMES[c['ID']]})",
        'R': f"{c['R']} ({DA_NAMES[c['R']]})",
        'w(S)': int(c['w'])
    }
    for c in admissible
])

print(df_admissible.to_string(index=False))

Espaco admissivel: 144 de 144 configuracoes
Reducao: 0.0%
  w(S) = 3: 54 configuracoes
  w(S) = 2: 90 configuracoes

  ID                     D                        I                     ID_                           R  w(S)
 S01      D1 (model-based)   I1 (output comparison)        ID1 (rule-based)              R1 (exclusion)     3
 S02      D1 (model-based)   I1 (output comparison)        ID1 (rule-based) R2 (weight reconfiguration)     3
 S03      D1 (model-based)   I1 (output comparison)        ID1 (rule-based)                 R3 (bypass)     3
 S04      D1 (model-based)   I1 (output comparison)       ID2 (statistical)              R1 (exclusion)     3
 S05      D1 (model-based)   I1 (output comparison)       ID2 (statistical) R2 (weight reconfiguration)     3
 S06      D1 (model-based)   I1 (output comparison)       ID2 (statistical)                 R3 (bypass)     2
 S07      D1 (model-based)   I1 (output comparison) ID3 (ML identification)              R1 (exclusion)     3
 S0

### Resumo do espaco de solucoes


In [112]:
n_total = len(all_configs)
n_admissible = len(admissible)
n_w3 = sum(1 for c in admissible if c['w'] == 3)
n_w2 = sum(1 for c in admissible if c['w'] == 2)

print("Resumo do espaco de solucoes")
print("-" * 40)
print(f"Espaco bruto:          {n_total} configuracoes (4 x 4 x 3 x 3)")
print(f"Threshold:             w(S) >= 2")
print(f"Espaco admissivel:     {n_admissible} configuracoes")
print(f"Reducao:               {100*(1 - n_admissible/n_total):.1f}%")
print(f"  w(S) = 3: {n_w3} configuracoes (todos os pares plenamente compativeis)")
print(f"  w(S) = 2: {n_w2} configuracoes (ao menos um par com restricao moderada)")
print()
print("Interpretacao: CCM nao elimina nenhuma configuracao.")
print("A diferenciacao e feita pelos metodos de selecao.")

Resumo do espaco de solucoes
----------------------------------------
Espaco bruto:          144 configuracoes (4 x 4 x 3 x 3)
Threshold:             w(S) >= 2
Espaco admissivel:     144 configuracoes
Reducao:               0.0%
  w(S) = 3: 54 configuracoes (todos os pares plenamente compativeis)
  w(S) = 2: 90 configuracoes (ao menos um par com restricao moderada)

Interpretacao: CCM nao elimina nenhuma configuracao.
A diferenciacao e feita pelos metodos de selecao.


### Metodos de Selecao

Quatro metodos implementados sobre o espaco admissivel (144 configuracoes).

Cada metodo expoe a mesma interface:
- Input: lista de configuracoes, cenario, Vetor A
- Output: ranking ou conjunto Pareto-eficiente

#### Scoring Linear Ponderado

Formulacao (Levin 2012, Secao 3.5 — Multiple Choice Problem):

$$Q(S) = \sum_{m \in \text{MODULES}} w_m \cdot \sum_{c \in \text{CRITERIA}} w_c \cdot \text{score}(da_m, c)$$

Onde:
- $w_m$: peso do modulo $m$ no cenario (homogeneo = 0.25)
- $w_c$: peso do criterio $c$ no cenario
- $\text{score}(da_m, c)$: score da DA selecionada para o modulo $m$ no criterio $c$ (Vetor A)

Output: escalar $Q(S)$ por configuracao — ranking por valor decrescente.

In [113]:
def scoring_linear(admissible, scenario, vetor_a):
    w_mod = scenario['module_weights']
    w_crit = scenario['criteria_weights']
    results = []

    for cfg in admissible:
        q = 0.0
        for mod in MODULES:
            da = cfg[mod]
            da_score = sum(
                w_crit[c] * vetor_a.loc[da, c]
                for c in CRITERIA
            )
            q += w_mod[mod] * da_score
        results.append({'id': cfg['id'], 'Q': round(q, 4)})

    results.sort(key=lambda x: x['Q'], reverse=True)
    for rank, r in enumerate(results, 1):
        r['rank'] = rank
    return results


# Executar para todos os cenarios
print("Scoring Linear Ponderado -- Top 10 por cenario")
print("=" * 60)

scoring_results = {}
for sc_id, sc in SCENARIOS.items():
    res = scoring_linear(admissible, sc, vetor_a)
    scoring_results[sc_id] = res
    print(f"\n{sc_id} -- {sc['name']}")
    print(f"{'Rank':<6} {'ID':<6} {'Q':>6}  {'D':<6} {'I':<6} {'ID':<6} {'R':<6}")
    print("-" * 55)
    cfg_map = {c['id']: c for c in admissible}
    for r in res[:10]:
        cfg = cfg_map[r['id']]
        print(f"{r['rank']:<6} {r['id']:<6} {r['Q']:>6.4f}  "
              f"{cfg['D']:<6} {cfg['I']:<6} {cfg['ID']:<6} {cfg['R']:<6}")

Scoring Linear Ponderado -- Top 10 por cenario

SC1 -- Uniforme
Rank   ID          Q  D      I      ID     R     
-------------------------------------------------------
1      S46    3.7500  D2     I2     ID1    R1    
2      S48    3.7500  D2     I2     ID1    R3    
3      S10    3.7000  D1     I2     ID1    R1    
4      S12    3.7000  D1     I2     ID1    R3    
5      S37    3.6500  D2     I1     ID1    R1    
6      S39    3.6500  D2     I1     ID1    R3    
7      S01    3.6000  D1     I1     ID1    R1    
8      S03    3.6000  D1     I1     ID1    R3    
9      S47    3.6000  D2     I2     ID1    R2    
10     S11    3.5500  D1     I2     ID1    R2    

SC2 -- Segurança / certificação
Rank   ID          Q  D      I      ID     R     
-------------------------------------------------------
1      S46    3.8750  D2     I2     ID1    R1    
2      S48    3.8750  D2     I2     ID1    R3    
3      S10    3.8500  D1     I2     ID1    R1    
4      S12    3.8500  D1     I2     ID1  

#### Metodo do Ponto Ideal

Calcula a distancia de cada configuracao ao ponto ideal S^o.

$S^o$: para cada modulo, a DA com maior score ponderado pelos criterios.

$$S^* = \arg\min_{S \in \{S_a\}} \rho(S, S^o)$$

$$\rho(S, S^o) = \sqrt{\sum_{m \in \text{MODULES}} w_m \cdot \sum_{c \in \text{CRITERIA}} w_c \cdot (score^o_c - score(da_m, c))^2}$$

Onde $S^o$ e a configuracao ideal formada pela melhor DA de cada modulo
por criterio ponderado. Output: distancia $\rho$ por configuracao --
ranking por valor crescente (menor distancia = melhor).

In [114]:

def ideal_point(admissible, scenario, vetor_a):
    w_mod = scenario['module_weights']
    w_crit = scenario['criteria_weights']

    # Calcular ponto ideal: melhor DA por modulo (maior score ponderado)
    ideal = {}
    for mod in MODULES:
        best_da = max(
            MODULE_VALUES[mod],
            key=lambda da: sum(w_crit[c] * vetor_a.loc[da, c] for c in CRITERIA)
        )
        ideal[mod] = {c: vetor_a.loc[best_da, c] for c in CRITERIA}

    results = []
    for cfg in admissible:
        rho_sq = 0.0
        for mod in MODULES:
            da = cfg[mod]
            for c in CRITERIA:
                diff = ideal[mod][c] - vetor_a.loc[da, c]
                rho_sq += w_mod[mod] * w_crit[c] * (diff ** 2)
        rho = round(rho_sq ** 0.5, 4)
        results.append({'id': cfg['id'], 'rho': rho})

    results.sort(key=lambda x: x['rho'])
    for rank, r in enumerate(results, 1):
        r['rank'] = rank
    return results


# Executar para todos os cenarios
print("Ideal Point -- Top 10 por cenario")
print("=" * 60)

ideal_results = {}
for sc_id, sc in SCENARIOS.items():
    res = ideal_point(admissible, sc, vetor_a)
    ideal_results[sc_id] = res
    print(f"\n{sc_id} -- {sc['name']}")
    print(f"{'Rank':<6} {'ID':<6} {'rho':>7}  {'D':<6} {'I':<6} {'ID':<6} {'R':<6}")
    print("-" * 55)
    cfg_map = {c['id']: c for c in admissible}
    for r in res[:10]:
        cfg = cfg_map[r['id']]
        print(f"{r['rank']:<6} {r['id']:<6} {r['rho']:>7.4f}  "
              f"{cfg['D']:<6} {cfg['I']:<6} {cfg['ID']:<6} {cfg['R']:<6}")

Ideal Point -- Top 10 por cenario

SC1 -- Uniforme
Rank   ID         rho  D      I      ID     R     
-------------------------------------------------------
1      S46     0.0000  D2     I2     ID1    R1    
2      S48     0.0000  D2     I2     ID1    R3    
3      S10     0.2236  D1     I2     ID1    R1    
4      S12     0.2236  D1     I2     ID1    R3    
5      S37     0.3162  D2     I1     ID1    R1    
6      S39     0.3162  D2     I1     ID1    R3    
7      S01     0.3873  D1     I1     ID1    R1    
8      S03     0.3873  D1     I1     ID1    R3    
9      S47     0.5000  D2     I2     ID1    R2    
10     S11     0.5477  D1     I2     ID1    R2    

SC2 -- Segurança / certificação
Rank   ID         rho  D      I      ID     R     
-------------------------------------------------------
1      S46     0.0000  D2     I2     ID1    R1    
2      S48     0.0000  D2     I2     ID1    R3    
3      S10     0.1581  D1     I2     ID1    R1    
4      S12     0.1581  D1     I2     ID

#### Pareto-Based Morphological Analysis

Uma configuracao $S$ e Pareto-eficiente se nao existe $S'$ tal que:

$$\forall c \in \text{CRITERIA}: f_c(S') \geq f_c(S) \quad \text{e} \quad \exists c: f_c(S') > f_c(S)$$

Onde $f_c(S)$ e o score agregado da configuracao $S$ no criterio $c$:

$$f_c(S) = \sum_{m \in \text{MODULES}} w_m \cdot \text{score}(da_m, c)$$

Output: conjunto de configuracoes nao-dominadas (front Pareto).
O Pareto-based MA nao depende de cenario para determinar dominancia --
os pesos dos modulos entram apenas no calculo de $f_c(S)$.

In [115]:
def compute_criteria_scores(admissible, scenario, vetor_a):
    w_mod = scenario['module_weights']
    scores = {}
    for cfg in admissible:
        scores[cfg['id']] = {
            c: sum(w_mod[mod] * vetor_a.loc[cfg[mod], c] for mod in MODULES)
            for c in CRITERIA
        }
    return scores


def pareto_based_ma(admissible, scenario, vetor_a):
    crit_scores = compute_criteria_scores(admissible, scenario, vetor_a)
    ids = [cfg['id'] for cfg in admissible]
    dominated = set()

    for i, s1 in enumerate(ids):
        for s2 in ids:
            if s1 == s2:
                continue
            # Verificar se s1 e dominada por s2
            geq_all = all(crit_scores[s2][c] >= crit_scores[s1][c] for c in CRITERIA)
            gt_one  = any(crit_scores[s2][c] >  crit_scores[s1][c] for c in CRITERIA)
            if geq_all and gt_one:
                dominated.add(s1)
                break

    pareto_front = [cfg for cfg in admissible if cfg['id'] not in dominated]
    return pareto_front, crit_scores


# Executar para todos os cenarios
print("Pareto-Based MA -- Front Pareto por cenario")
print("=" * 60)

pareto_results = {}
for sc_id, sc in SCENARIOS.items():
    front, crit_scores = pareto_based_ma(admissible, sc, vetor_a)
    pareto_results[sc_id] = {'front': front, 'crit_scores': crit_scores}

    print(f"\n{sc_id} -- {sc['name']}")
    print(f"Front Pareto: {len(front)} de {len(admissible)} configuracoes")
    print(f"{'ID':<6} {'D':<6} {'I':<6} {'ID':<6} {'R':<6} "
          f"{'f_C1':>6} {'f_C2':>6} {'f_C3':>6} {'f_C4':>6} {'f_C5':>6}")
    print("-" * 70)
    for cfg in sorted(front, key=lambda x: x['id']):
        cs = crit_scores[cfg['id']]
        print(f"{cfg['id']:<6} {cfg['D']:<6} {cfg['I']:<6} "
              f"{cfg['ID']:<6} {cfg['R']:<6} "
              f"{cs['C1']:>6.3f} {cs['C2']:>6.3f} {cs['C3']:>6.3f} "
              f"{cs['C4']:>6.3f} {cs['C5']:>6.3f}")

Pareto-Based MA -- Front Pareto por cenario

SC1 -- Uniforme
Front Pareto: 4 de 144 configuracoes
ID     D      I      ID     R        f_C1   f_C2   f_C3   f_C4   f_C5
----------------------------------------------------------------------
S46    D2     I2     ID1    R1      4.000  4.000  4.000  4.000  2.750
S47    D2     I2     ID1    R2      3.750  3.750  3.750  3.750  3.000
S48    D2     I2     ID1    R3      4.000  4.000  4.000  4.000  2.750
S50    D2     I2     ID2    R2      3.500  3.250  3.500  3.500  3.250

SC2 -- Segurança / certificação
Front Pareto: 4 de 144 configuracoes
ID     D      I      ID     R        f_C1   f_C2   f_C3   f_C4   f_C5
----------------------------------------------------------------------
S46    D2     I2     ID1    R1      4.000  4.000  4.000  4.000  2.750
S47    D2     I2     ID1    R2      3.750  3.750  3.750  3.750  3.000
S48    D2     I2     ID1    R3      4.000  4.000  4.000  4.000  2.750
S50    D2     I2     ID2    R2      3.500  3.250  3.500  3.5

#### Hierarchical Morphological Multicriteria Design (HMMD)

O vetor de qualidade do sistema e definido como:

$$N(S) = (w(S);\, e(S))$$

Onde:
- $w(S) = \min_{j_1 \neq j_2} \text{CCM}(da_{j_1}, da_{j_2})$ — minimo das compatibilidades par-a-par
- $e(S) = (\eta_1, \eta_2, \eta_3, \eta_4)$ — vetor multiset contando DAs por nivel de prioridade, com $\sum_r \eta_r = m$ (numero de modulos)

**Prioridades ordinais das DAs por modulo:**
Calculadas por score ponderado pelos criterios do cenario (D07):

$$\text{score}(da, sc) = \sum_{c \in \text{CRITERIA}} w_c(sc) \cdot \text{score}(da, c)$$

A posicao ordinal dentro do modulo e determinada pelo ranking desse score
(posicao 1 = maior score). Empates recebem a mesma posicao (D05).
Dimensao $k = 4$ global para todos os modulos (D06).

**Criterio de Pareto-eficiencia sobre $N(S)$:**
$S$ e nao-dominada se nao existe $S'$ tal que $N(S') \succ N(S)$,
onde $\succ$ denota dominancia lexicografica sobre $(w(S); e(S))$.

**Problema de otimizacao (Levin 2015, Secao 2.2.7):**

$$\max\, e(S), \quad \max\, w(S), \quad \text{s.t.} \quad w(S) \geq 0.5$$

In [116]:
def compute_priorities(scenario, vetor_a):
    w_crit = scenario['criteria_weights']
    priorities = {}

    for mod in MODULES:
        das = MODULE_VALUES[mod]
        scores = {
            da: sum(w_crit[c] * vetor_a.loc[da, c] for c in CRITERIA)
            for da in das
        }
        # Ranking com empates na mesma posicao (dense ranking)
        sorted_scores = sorted(set(scores.values()), reverse=True)
        rank_map = {s: i+1 for i, s in enumerate(sorted_scores)}
        priorities[mod] = {da: rank_map[scores[da]] for da in das}

    return priorities


def compute_e(config, priorities, k=4):
    e = [0] * k
    for mod in MODULES:
        da = config[mod]
        rank = priorities[mod][da]
        if rank <= k:
            e[rank - 1] += 1
    return tuple(e)


def hmmd_dominates(n1, n2):
    w1, e1 = n1
    w2, e2 = n2

    # Comparacao de w
    w_geq = w1 >= w2
    w_gt  = w1 > w2

    # Comparacao de e pelo poset (lexicografico)
    e_geq = e1 >= e2  # tuple comparison -- lexicografico
    e_gt  = e1 > e2

    return (w_geq and e_geq) and (w_gt or e_gt)


def hmmd(admissible, scenario, vetor_a, k=4):
    priorities = compute_priorities(scenario, vetor_a)

    # Calcular N(S) para cada configuracao
    ns = {}
    for cfg in admissible:
        w = cfg['w']
        e = compute_e(cfg, priorities, k)
        ns[cfg['id']] = (w, e)

    # Identificar solucoes nao-dominadas
    dominated = set()
    ids = [cfg['id'] for cfg in admissible]
    for s1 in ids:
        for s2 in ids:
            if s1 == s2:
                continue
            if hmmd_dominates(ns[s2], ns[s1]):
                dominated.add(s1)
                break

    pareto_hmmd = [cfg for cfg in admissible if cfg['id'] not in dominated]
    return pareto_hmmd, ns, priorities


# Executar para todos os cenarios
print("HMMD -- Front Pareto por cenario")
print("=" * 60)

hmmd_results = {}
for sc_id, sc in SCENARIOS.items():
    front, ns, priorities = hmmd(admissible, sc, vetor_a)
    hmmd_results[sc_id] = {'front': front, 'ns': ns, 'priorities': priorities}

    print(f"\n{sc_id} -- {sc['name']}")
    print(f"Front HMMD: {len(front)} de {len(admissible)} configuracoes")
    print(f"{'ID':<6} {'D':<6} {'I':<6} {'ID':<6} {'R':<6} "
          f"{'w(S)':>6} {'e(S)':<20} {'N(S)'}")
    print("-" * 75)
    for cfg in sorted(front, key=lambda x: x['id']):
        n = ns[cfg['id']]
        print(f"{cfg['id']:<6} {cfg['D']:<6} {cfg['I']:<6} "
              f"{cfg['ID']:<6} {cfg['R']:<6} "
              f"{n[0]:>6} {str(n[1]):<20} ({n[0]}, {n[1]})")

HMMD -- Front Pareto por cenario

SC1 -- Uniforme
Front HMMD: 1 de 144 configuracoes
ID     D      I      ID     R        w(S) e(S)                 N(S)
---------------------------------------------------------------------------
S46    D2     I2     ID1    R1        3.0 (4, 0, 0, 0)         (3.0, (4, 0, 0, 0))

SC2 -- Segurança / certificação
Front HMMD: 1 de 144 configuracoes
ID     D      I      ID     R        w(S) e(S)                 N(S)
---------------------------------------------------------------------------
S46    D2     I2     ID1    R1        3.0 (4, 0, 0, 0)         (3.0, (4, 0, 0, 0))

SC3 -- Cobertura / desempenho
Front HMMD: 1 de 144 configuracoes
ID     D      I      ID     R        w(S) e(S)                 N(S)
---------------------------------------------------------------------------
S46    D2     I2     ID1    R1        3.0 (4, 0, 0, 0)         (3.0, (4, 0, 0, 0))

SC4 -- Eficiência de recursos
Front HMMD: 1 de 144 configuracoes
ID     D      I      ID     R     

In [117]:
# Diagnostico de prioridades por cenario e modulo
for sc_id, sc in SCENARIOS.items():
    w_crit = sc['criteria_weights']
    print(f"\n{sc_id} -- {sc['name']}")
    print(f"{'DA':<6} {'Modulo':<8} {'Score':>7} {'Rank':>6}")
    print("-" * 35)
    for mod in MODULES:
        das = MODULE_VALUES[mod]
        scores = {
            da: round(sum(w_crit[c] * vetor_a.loc[da, c] for c in CRITERIA), 4)
            for da in das
        }
        sorted_scores = sorted(set(scores.values()), reverse=True)
        rank_map = {s: i+1 for i, s in enumerate(sorted_scores)}
        for da in das:
            print(f"{da:<6} {mod:<8} {scores[da]:>7.4f} {rank_map[scores[da]]:>6}")


SC1 -- Uniforme
DA     Modulo     Score   Rank
-----------------------------------
D1     D         3.6000      2
D2     D         3.8000      1
D3     D         1.8000      4
D4     D         2.2000      3
I1     I         3.4000      2
I2     I         3.8000      1
I3     I         2.6000      3
I4     I         2.6000      3
ID1    ID        3.6000      1
ID2    ID        2.8000      2
ID3    ID        1.8000      3
R1     R         3.8000      1
R2     R         3.2000      2
R3     R         3.8000      1

SC2 -- Segurança / certificação
DA     Modulo     Score   Rank
-----------------------------------
D1     D         3.8000      2
D2     D         3.9000      1
D3     D         1.6500      4
D4     D         2.3500      3
I1     I         3.7000      2
I2     I         3.9000      1
I3     I         2.8000      3
I4     I         2.8000      3
ID1    ID        3.8000      1
ID2    ID        2.9000      2
ID3    ID        1.6500      3
R1     R         3.9000      1
R2     R  

## Experimentos Consolidados

Execução dos 4 métodos de seleção sobre o espaço admissível (144 configurações).
Inputs: Vetor A escala 1-4 (consolidado de 4 LLMs, pós-revisão) · CCM escala {0,1,2,3}.

**Métodos:**
- Scoring linear ponderado: ranking por Q(S) decrescente
- Ideal point: ranking por distância ρ crescente ao ponto ideal
- Pareto-based MA: conjunto não-dominado sobre vetores de critérios
- HMMD: conjunto não-dominado sobre N(S) = (w(S); e(S))

In [118]:
# Mapa de configuracoes para lookup rapido
cfg_map = {c['id']: c for c in admissible}

# 4.1 Tabela comparativa -- Scoring e Ideal Point por cenario
print("Secao 4 -- Comparacao de Rankings")
print("=" * 60)

for sc_id, sc in SCENARIOS.items():
    s_res = scoring_results[sc_id]
    i_res = ideal_results[sc_id]

    # Converter para dict rank por ID
    s_rank = {r['id']: r['rank'] for r in s_res}
    i_rank = {r['id']: r['rank'] for r in i_res}

    # Top 10 uniao (IDs que aparecem em qualquer top 10)
    top_ids = list(dict.fromkeys(
        [r['id'] for r in s_res[:10]] +
        [r['id'] for r in i_res[:10]]
    ))

    print(f"\n{sc_id} -- {sc['name']}")
    print(f"{'ID':<6} {'D':<6} {'I':<6} {'ID':<6} {'R':<4} "
          f"{'Scoring':>8} {'Q':>7} {'Ideal':>7} {'rho':>7} {'Pareto':>8} {'HMMD':>6}")
    print("-" * 72)

    # Pareto front IDs para este cenario
    pareto_ids = {c['id'] for c in pareto_results[sc_id]['front']}
    hmmd_ids = {c['id'] for c in hmmd_results[sc_id]['front']}

    s_dict = {r['id']: r for r in s_res}
    i_dict = {r['id']: r for r in i_res}

    for sid in top_ids:
        cfg = cfg_map[sid]
        s_r = s_rank.get(sid, '-')
        i_r = i_rank.get(sid, '-')
        q = s_dict[sid]['Q']
        rho = i_dict[sid]['rho']
        pareto = 'Pareto' if sid in pareto_ids else ''
        hmmd_flag = 'HMMD' if sid in hmmd_ids else ''
        print(f"{sid:<6} {cfg['D']:<6} {cfg['I']:<6} {cfg['ID']:<6} {cfg['R']:<4} "
              f"{s_r:>8} {q:>7.4f} {i_r:>7} {rho:>7.4f} {pareto:>8} {hmmd_flag:>6}")

Secao 4 -- Comparacao de Rankings

SC1 -- Uniforme
ID     D      I      ID     R     Scoring       Q   Ideal     rho   Pareto   HMMD
------------------------------------------------------------------------
S46    D2     I2     ID1    R1          1  3.7500       1  0.0000   Pareto   HMMD
S48    D2     I2     ID1    R3          2  3.7500       2  0.0000   Pareto       
S10    D1     I2     ID1    R1          3  3.7000       3  0.2236                
S12    D1     I2     ID1    R3          4  3.7000       4  0.2236                
S37    D2     I1     ID1    R1          5  3.6500       5  0.3162                
S39    D2     I1     ID1    R3          6  3.6500       6  0.3162                
S01    D1     I1     ID1    R1          7  3.6000       7  0.3873                
S03    D1     I1     ID1    R3          8  3.6000       8  0.3873                
S47    D2     I2     ID1    R2          9  3.6000       9  0.5000   Pareto       
S11    D1     I2     ID1    R2         10  3.5500      1

## Analise Comparativa

Correlacao de Spearman entre metodos, identificacao de divergencias
e achados transversais.

In [119]:
from scipy.stats import spearmanr

print("Secao 5 -- Analise Comparativa")
print("=" * 60)

# 5.1 Correlacao de Spearman -- Scoring vs Ideal Point
print("\n5.1 Correlacao de Spearman -- Scoring vs Ideal Point")
print("-" * 50)
print(f"{'Cenario':<8} {'rho':>8} {'p':>10}")
print("-" * 30)

for sc_id in SCENARIOS:
    s_res = scoring_results[sc_id]
    i_res = ideal_results[sc_id]

    # Alinhar rankings pelo ID
    ids = [r['id'] for r in s_res]
    s_ranks = [r['rank'] for r in s_res]
    i_rank_map = {r['id']: r['rank'] for r in i_res}
    i_ranks = [i_rank_map[sid] for sid in ids]

    rho, p = spearmanr(s_ranks, i_ranks)
    print(f"{sc_id:<8} {rho:>8.4f} {p:>10.4f}")

# 5.2 Correlacao de Spearman -- entre cenarios (scoring)
print("\n5.2 Correlacao de Spearman -- Scoring entre cenarios")
print("-" * 50)
sc_ids = list(SCENARIOS.keys())
print(f"{'':>8}", end='')
for sc in sc_ids:
    print(f"{sc:>8}", end='')
print()

for sc1 in sc_ids:
    print(f"{sc1:<8}", end='')
    r1 = {r['id']: r['rank'] for r in scoring_results[sc1]}
    for sc2 in sc_ids:
        r2 = {r['id']: r['rank'] for r in scoring_results[sc2]}
        ids = list(r1.keys())
        rho, _ = spearmanr([r1[i] for i in ids], [r2[i] for i in ids])
        print(f"{rho:>8.4f}", end='')
    print()

# 5.3 Divergencias entre metodos
print("\n5.3 Divergencias identificadas")
print("-" * 50)

for sc_id, sc in SCENARIOS.items():
    s_top10 = {r['id'] for r in scoring_results[sc_id][:10]}
    i_top10 = {r['id'] for r in ideal_results[sc_id][:10]}
    pareto_ids = {c['id'] for c in pareto_results[sc_id]['front']}

    # Em Pareto mas nao no top 10 scoring
    pareto_not_scoring = pareto_ids - s_top10
    # No top 10 scoring mas nao no Pareto
    scoring_not_pareto = s_top10 - pareto_ids

    print(f"\n{sc_id} -- {sc['name']}")
    if pareto_not_scoring:
        print(f"  Pareto-eficiente mas fora do top 10 scoring: "
              f"{', '.join(sorted(pareto_not_scoring))}")
        for sid in sorted(pareto_not_scoring):
            cfg = cfg_map[sid]
            cs = pareto_results[sc_id]['crit_scores'][sid]
            print(f"    {sid}: D={cfg['D']} I={cfg['I']} "
                  f"ID={cfg['ID']} R={cfg['R']}")
            print(f"    vetores C1-C5: "
                  f"{[round(float(cs[c]),3) for c in CRITERIA]}")
    if scoring_not_pareto:
        print(f"  Top 10 scoring mas nao Pareto-eficiente: "
              f"{', '.join(sorted(scoring_not_pareto))}")
    if not pareto_not_scoring and not scoring_not_pareto:
        print("  Sem divergencias relevantes.")

# 5.4 Achados transversais
print("\n5.4 Achados transversais")
print("-" * 50)

# Estabilidade do ranking
rho_min = 1.0
rho_min_pair = ('', '')
for i, sc1 in enumerate(sc_ids):
    for sc2 in sc_ids[i+1:]:
        r1 = {r['id']: r['rank'] for r in scoring_results[sc1]}
        r2 = {r['id']: r['rank'] for r in scoring_results[sc2]}
        ids = list(r1.keys())
        rho, _ = spearmanr([r1[i] for i in ids], [r2[i] for i in ids])
        if rho < rho_min:
            rho_min = rho
            rho_min_pair = (sc1, sc2)

print(f"Spearman minimo entre cenarios (scoring): "
      f"{rho_min:.4f} ({rho_min_pair[0]} x {rho_min_pair[1]})")

# Calcular rank medio e maximo por configuracao
rank_stats = {}
for sid in [c['id'] for c in admissible]:
    ranks = [r['rank'] for sc in sc_ids
             for r in scoring_results[sc] if r['id'] == sid]
    rank_stats[sid] = {
        'mean': sum(ranks) / len(ranks),
        'max': max(ranks),
        'ranks': ranks
    }

# Ordenar por rank medio, desempate por rank maximo
sorted_by_robustness = sorted(
    rank_stats.items(),
    key=lambda x: (x[1]['mean'], x[1]['max'])
)

# Pareto e HMMD em todos os cenarios
pareto_todos = set.intersection(*[
    {c['id'] for c in pareto_results[sc]['front']}
    for sc in sc_ids
])
hmmd_todos = set.intersection(*[
    {c['id'] for c in hmmd_results[sc]['front']}
    for sc in sc_ids
])

print("Top 5 configuracoes por robustez (rank medio no scoring):")
print(f"{'ID':<6} {'D':<6} {'I':<6} {'ID':<6} {'R':<6} "
      f"{'Ranks SC1-SC4':<20} {'Media':>6} {'Max':>5} {'Pareto':>8} {'HMMD':>6}")
print("-" * 80)
for sid, stats in sorted_by_robustness[:5]:
    cfg = cfg_map[sid]
    pareto_flag = 'Pareto' if sid in pareto_todos else ''
    hmmd_flag = 'HMMD' if sid in hmmd_todos else ''
    print(f"{sid:<6} {cfg['D']:<6} {cfg['I']:<6} {cfg['ID']:<6} {cfg['R']:<6} "
          f"{str(stats['ranks']):<20} {stats['mean']:>6.2f} {stats['max']:>5} "
          f"{pareto_flag:>8} {hmmd_flag:>6}")

# Trade-off identificado
print(f"\nTrade-off C5 vs C1-C4:")
print(f"  S50 (D2, I2, ID2, R2) -- Pareto-eficiente em todos os cenarios")
cs_s50 = pareto_results['SC1']['crit_scores']['S50']
print(f"  Vetor C1-C5: {[round(float(cs_s50[c]), 3) for c in CRITERIA]}")
print(f"  C5 maior que S46/S48, mas C1-C4 menores.")
print(f"  Nao aparece no top 10 do scoring em nenhum cenario.")

Secao 5 -- Analise Comparativa

5.1 Correlacao de Spearman -- Scoring vs Ideal Point
--------------------------------------------------
Cenario       rho          p
------------------------------
SC1        0.9579     0.0000
SC2        0.9507     0.0000
SC3        0.9204     0.0000
SC4        0.9600     0.0000

5.2 Correlacao de Spearman -- Scoring entre cenarios
--------------------------------------------------
             SC1     SC2     SC3     SC4
SC1       1.0000  0.9924  0.9767  0.9895
SC2       0.9924  1.0000  0.9505  0.9815
SC3       0.9767  0.9505  1.0000  0.9548
SC4       0.9895  0.9815  0.9548  1.0000

5.3 Divergencias identificadas
--------------------------------------------------

SC1 -- Uniforme
  Pareto-eficiente mas fora do top 10 scoring: S50
    S50: D=D2 I=I2 ID=ID2 R=R2
    vetores C1-C5: [3.5, 3.25, 3.5, 3.5, 3.25]
  Top 10 scoring mas nao Pareto-eficiente: S01, S03, S10, S11, S12, S37, S39

SC2 -- Segurança / certificação
  Pareto-eficiente mas fora do top 10 s

---
## Referencias


Bader, K., Lussier, B., & Schön, W. (2017a). A fault tolerant architecture
for data fusion: A real application of Kalman filters for mobile robot
localization. *Robotics and Autonomous Systems*, *88*, 11-23.
https://doi.org/10.1016/j.robot.2016.12.001

Bader, K., Lussier, B., & Schön, W. (2017b). Fault tolerance from formal
analysis of a data fusion mechanism. In *2017 First IEEE International
Conference on Robotic Computing (IRC)* (pp. 69-74). IEEE.
https://doi.org/10.1109/IRC.2017.58

Ceccarelli, A., & Secci, F. (2023). RGB cameras failures and their effects
in autonomous driving applications. *IEEE Transactions on Dependable and
Secure Computing*, *20*(4), 2731-2745.
https://doi.org/10.1109/TDSC.2022.3156941

Conejo, C., Puig, V., Morcego, B., Navas, F., & Milanes, V. (2025).
Enhancing safety in autonomous vehicles using zonotopic LPV-EKF for fault
detection and isolation in state estimation. *Control Engineering Practice*,
*156*, 106192. https://doi.org/10.1016/j.conengprac.2024.106192

Elhoseny, M., Rao, D. D., Veerasamy, B. D., Alduaiji, N., Shreyas, J., &
Shukla, P. K. (2024). Deep learning algorithm for optimized sensor data
fusion in fault diagnosis and tolerance. *International Journal of
Computational Intelligence Systems*, *17*, 299.
https://doi.org/10.1007/s44196-024-00692-5

Emzivat, Y., Delahaye, B., Roux, O. H., & El Najjar, M. E. (2018). A formal
approach for the design of a dependable perception system for autonomous
vehicles. In *2018 IEEE Intelligent Vehicles Symposium (IV)* (pp. 1297-1304).
IEEE. https://doi.org/10.1109/IVS.2018.8500412

Erhan, L., Ndubuaku, M., Di Mauro, M., Song, W., Chen, M., Fortino, G.,
Bagdasar, O., & Liotta, A. (2021). Smart anomaly detection in sensor systems:
A multi-perspective review. *Information Fusion*, *67*, 64-79.
https://doi.org/10.1016/j.inffus.2020.10.001

Gao, Z., Cecati, C., & Ding, S. X. (2015a). A survey of fault diagnosis and
fault-tolerant techniques -- Part I: Fault diagnosis with model-based and
signal-based approaches. *IEEE Transactions on Industrial Electronics*,
*62*(6), 3757-3767. https://doi.org/10.1109/TIE.2015.2417570

Gao, Z., Cecati, C., & Ding, S. X. (2015b). A survey of fault diagnosis and
fault-tolerant techniques -- Part II: Fault diagnosis with knowledge-based
and hybrid/active approaches. *IEEE Transactions on Industrial Electronics*,
*62*(6), 3768-3774. https://doi.org/10.1109/TIE.2015.2419013

Goelles, T., Schlager, B., & Muckenhuber, S. (2020). Fault detection,
isolation, identification and recovery (FDIIR) methods for automotive
perception sensors including a detailed literature survey for lidar.
*Sensors*, *20*(13), 3662. https://doi.org/10.3390/s20133662

Grubmuller, S., Stettinger, G., Sotelo, M. A., & Watzenig, D. (2019).
Fault-tolerant environmental perception architecture for robust automated
driving. In *2019 IEEE International Conference on Connected Vehicles and
Expo (ICCVE)* (pp. 1-6). IEEE.
https://doi.org/10.1109/ICCVE45908.2019.8965112

Hao, Y., Ding, Y., Wang, G., Zhou, Y., & Jia, X. (2019). A fault-tolerant
data fusion method based on decentralized Kalman filter for redundant sensor
configuration. In *Proceedings of the 38th Chinese Control Conference*
(pp. 4486-4491). IEEE. https://doi.org/10.23919/ChiCC.2019.8856985

Hou, W., Li, W., & Li, P. (2023). Fault diagnosis of the autonomous driving
perception system based on information fusion. *Sensors*, *23*(11), 5110.
https://doi.org/10.3390/s23115110

Koopman, P., & Wagner, M. (2017). Autonomous vehicle safety: An
interdisciplinary challenge. *IEEE Intelligent Transportation Systems
Magazine*, *9*(1), 90-96. https://doi.org/10.1109/MITS.2016.2583491

Lee, J., Kang, H., Jeon, Y., Cho, J., Kim, S., & Jo, K. (2024). Sensor
fusion-based classification fault-tolerant system for detected objects in
autonomous cars. *IEEE Transactions on Intelligent Vehicles*.
https://doi.org/10.1109/TIV.2024.3361001

Mendonca, R., Rodrigues, J., Rodrigues, R., & Mendonca, R. (2023). Enhancing
the reliability of perception systems using N-version programming and
rejuvenation. In *2023 IEEE 28th Pacific Rim International Symposium on
Dependable Computing (PRDC)* (pp. 11-20). IEEE.
https://doi.org/10.1109/PRDC59308.2023.00012

Min, H., Fang, Y., Wu, X., Lei, X., Chen, S., Teixeira, R., Zhu, B., Zhao,
X., & Xu, Z. (2023). A fault diagnosis framework for autonomous vehicles with
sensor self-diagnosis. *Expert Systems with Applications*, *224*, 120002.
https://doi.org/10.1016/j.eswa.2023.120002

Nitsch, J., Itkina, M., Senanayake, R., Nieto, J., Schmidt, M., Siegwart,
R., Kochenderfer, M. J., & Cadena, C. (2021). Out-of-distribution detection
for automotive perception. In *2021 IEEE International Intelligent
Transportation Systems Conference (ITSC)* (pp. 2938-2943). IEEE.
https://doi.org/10.1109/ITSC48978.2021.9564545

Sinha, R., Schmerling, E., & Pavone, M. (2023). Closing the loop on runtime
monitors with fallback-safe MPC. *arXiv preprint arXiv:2309.08603*.
https://doi.org/10.48550/arXiv.2309.08603

Tian, H., Ding, W., Han, X., Wu, G., Guo, A., Zhang, J., Chen, W., Wei, J.,
& Zhang, T. (2025). Testing the fault-tolerance of multi-sensor fusion
perception in autonomous driving systems. *Proceedings of the ACM on
Software Engineering*, *2*(ISSTA), Article ISSTA035.
https://doi.org/10.1145/3728910

van Wyk, F., Wang, Y., Khojandi, A., & Masoud, N. (2020). Real-time sensor
anomaly detection and identification in automated vehicles. *IEEE
Transactions on Intelligent Transportation Systems*, *21*(3), 1264-1276.
https://doi.org/10.1109/TITS.2019.2906038

Werling, M., Faller, R., Betz, W., & Straub, D. (2025). Safety integrity
framework for automated driving. *arXiv preprint arXiv:2503.20544*.
https://doi.org/10.48550/arXiv.2503.20544

Levin, M. Sh. (2012). Morphological methods for design of modular systems:
A survey. *arXiv preprint arXiv:1201.1712*.

Levin, M. Sh. (2015). *Modular System Design and Evaluation*. Springer.

Ritchey, T. (2015). Principles of cross-consistency assessment in general
morphological modelling. *Acta Morphologica Generalis*, *4*(2).

Roy, B. (1996). *Multicriteria Methodology for Decision Aiding*.
Kluwer Academic Publishers.


### Processo de avaliacao -- LLMs

#### Cross-Consistency Matrix

| LLM | Link |
|---|---|
| Claude Opus 4.7 | https://claude.ai/share/73f04588-6a73-4535-b7d7-0a9c12954615 |
| Gemini 2.0 Flash | https://gemini.google.com/share/453f65692f63 |
| GPT-5.3 | https://chatgpt.com/share/69eb6dea-fe68-83e9-b521-7485b31f8fa1 |
| DeepSeek-R1 | https://chat.deepseek.com/share/jodbdp0dvma5cbtwn8 |